In [2]:
from __future__ import annotations

import json
import hashlib
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import Lasso, ElasticNet, Ridge, LassoLars
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import GroupKFold, ParameterGrid, StratifiedKFold
from joblib import Parallel, delayed
import os
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import f_regression

# Optional XGBoost candidate.
# If xgboost is not installed in the current environment, the xgboost model
# will be removed from MODEL_TYPES below to avoid crashing the whole run.
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False


In [3]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [4]:
# ==========================================
# Config
# ==========================================
DATA_ROOT = Path("../../DifferentCom_data_rebuild")
TP_DIR = DATA_ROOT / "tp_views"
BASE_DIR = DATA_ROOT / "base_patient_tables"

# ==========================================
# Experiment tag / output directory
# ==========================================
# Training version of paper-style single-omics:
#   1) uses TRAIN_IDS only
#   2) evaluates performance with outer CV OOF predictions
#   3) feature selection is performed inside each inner CV fold
#   4) f_regression / correlation univariate selection modes are supported
#   5) v11 uses targeted performance expansion with v9-compatible candidate-model pools
#   6) critical v9 routes are reproduced with the same candidate model sets, not single-model-only routes
#   7) final judgment must use OOF R², not train_r2 or final_full_train_r2
EXPERIMENT_TAG = "single_omics_train_paper_style_v11_rescue_expansion_v9compatible"

SAVE_DIR = DATA_ROOT / f"results_single_omics_train_only_{EXPERIMENT_TAG}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PATH = BASE_DIR / "target_by_patient.csv"

# A = proteomics, B = metabolomics, C = miRNA
COMBOS = ["A", "B", "C"]
TISSUES = ["csf", "ser"]
TIMEPOINTS = [24, 48, 72, 96, 120]

# Conservative model set for small-n single-omics.
# v11 keeps all model definitions available, but the actual run is
# controlled by the v9-compatible rescue expansion route table in the run cell.
MODEL_TYPES = [
    "ridge",
    "elasticnet",
    "pls",
    "svr_linear",
    "gbr",
    "xgboost",
]

# Keep the notebook runnable even when xgboost is not installed.
if not XGBOOST_AVAILABLE:
    print("[WARN] xgboost is not installed. Removing 'xgboost' from MODEL_TYPES.")
    MODEL_TYPES = [m for m in MODEL_TYPES if m != "xgboost"]


def get_candidate_models_for_single(tissue: str, combo: str, tp: int) -> list[str]:
    """
    v10 targeted OOF-focused routing based on the v9 single-omics summary.

    Main intent:
    - Keep ALL A/B/C combos and ALL 24/48/72/96/120h timepoints.
    - Use the v9 OOF winners as the primary route.
    - Keep a small number of backup models only where v9 showed useful signal.
    - Reduce noisy searches in weak B/C regions.
    """
    tissue = str(tissue).lower()
    combo = str(combo)
    tp = int(tp)

    if combo == "A":
        if tissue == "ser" and tp in [72, 96]:
            candidates = ["svr_linear", "ridge"]
        elif tissue == "csf" and tp == 72:
            candidates = ["svr_linear", "gbr", "ridge"]
        elif tissue == "csf" and tp == 96:
            candidates = ["ridge", "svr_linear"]
        else:
            candidates = ["ridge"]

    elif combo == "B":
        # B is kept mainly for complete A/B/C coverage.
        # v9 showed ridge as the only consistently useful B-family model.
        candidates = ["ridge"]

    elif combo == "C":
        if tissue == "ser" and tp == 24:
            # v9 overall best: ser C 24h + xgboost + f_regression_topk + top_k=40.
            candidates = ["xgboost", "ridge"]
        else:
            candidates = ["ridge"]

    else:
        candidates = ["ridge"]

    return [m for m in candidates if m in MODEL_TYPES]


def get_feature_selection_modes_for_single(tissue: str, combo: str, tp: int) -> list[str]:
    """
    v10 feature-selection routing.

    - A keeps both f_regression_topk and corr_topk because v9 showed both can win
      depending on tissue/TP, especially ser A 96h and csf A 72/96h.
    - ser C 24h keeps f_regression_topk as the primary XGBoost rescue route and
      corr_topk as a low-priority diagnostic route.
    - Other C routes use f_regression_topk primarily, with selected corr_topk checks
      in the explicit route table.
    - B remains f_regression_topk-only.
    """
    tissue = str(tissue).lower()
    combo = str(combo)
    tp = int(tp)

    if combo == "A":
        return ["f_regression_topk", "corr_topk"]

    if tissue == "ser" and combo == "C" and tp == 24:
        return ["f_regression_topk", "corr_topk"]

    return ["f_regression_topk"]


FEATURE_SELECTION_MODES_TO_RUN = [
    "f_regression_topk",
    "corr_topk",
]

# For ablation, change to ["omics_only", "light"].
CLINICAL_MODES_TO_RUN = ["light"]

# sparse filter: train 기준으로만 적용
MIN_OBS_FRAC = 0.50

# CV
N_SPLITS_OUTER = 5
N_SPLITS_INNER = 3
RANDOM_STATE = 42

# y-bin stratification for regression CV stability
USE_STRATIFIED_OUTER_CV = True
N_Y_BINS_FOR_OUTER_CV = 3
OUTER_CV_RANDOM_STATE = 42

# Model-selection stability penalty.
# best_score_for_selection = inner_oof_r2 - penalty * std(inner_fold_r2)
INNER_SELECTION_STD_PENALTY = 0.10

# ==========================================
# Leakage-free feature-wise weighting
# ==========================================
USE_FEATURE_WEIGHTING = True
FEATURE_WEIGHT_SCORE_MODE = "univariate_corr_soft_conservative"
FEATURE_WEIGHT_MIN = 1.0
FEATURE_WEIGHT_MAX = 1.20
FEATURE_WEIGHT_MIN_OBS = 8
FEATURE_WEIGHT_EXCLUDE_CLINICAL = True
FEATURE_WEIGHT_DEFAULT = 1.0

FEATURE_TRACE_DIR = SAVE_DIR / "feature_traces"
FEATURE_TRACE_DIR.mkdir(parents=True, exist_ok=True)

DETAIL_DIR = SAVE_DIR / "single_detail_outputs"
DETAIL_DIR.mkdir(parents=True, exist_ok=True)

print("SAVE_DIR:", SAVE_DIR.resolve())
print("DETAIL_DIR:", DETAIL_DIR.resolve())
print("EXPERIMENT_TAG:", EXPERIMENT_TAG)
print("MODEL_TYPES:", MODEL_TYPES)
print("MODEL_ROUTING_EXAMPLES:", {(t,c,tp): get_candidate_models_for_single(t,c,tp) for t in ["csf","ser"] for c in ["A","B","C"] for tp in [24,96]})
print("FEATURE_SELECTION_MODES_TO_RUN:", FEATURE_SELECTION_MODES_TO_RUN)
print("FEATURE_SELECTION_ROUTING_EXAMPLES:", {
    (t,c,tp): get_feature_selection_modes_for_single(t,c,tp)
    for t in ["csf","ser"] for c in ["A","B","C"] for tp in [24,72,120]
})
print("CLINICAL_MODES_TO_RUN:", CLINICAL_MODES_TO_RUN)
print("USE_FEATURE_WEIGHTING:", USE_FEATURE_WEIGHTING)
print("FEATURE_WEIGHT_SCORE_MODE:", FEATURE_WEIGHT_SCORE_MODE)
print("FEATURE_WEIGHT_RANGE:", (FEATURE_WEIGHT_MIN, FEATURE_WEIGHT_MAX))



SAVE_DIR: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/results_single_omics_train_only_single_omics_train_paper_style_v11_rescue_expansion_v9compatible
DETAIL_DIR: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/results_single_omics_train_only_single_omics_train_paper_style_v11_rescue_expansion_v9compatible/single_detail_outputs
EXPERIMENT_TAG: single_omics_train_paper_style_v11_rescue_expansion_v9compatible
MODEL_TYPES: ['ridge', 'elasticnet', 'pls', 'svr_linear', 'gbr', 'xgboost']
MODEL_ROUTING_EXAMPLES: {('csf', 'A', 24): ['ridge'], ('csf', 'A', 96): ['ridge', 'svr_linear'], ('csf', 'B', 24): ['ridge'], ('csf', 'B', 96): ['ridge'], ('csf', 'C', 24): ['ridge'], ('csf', 'C', 96): ['ridge'], ('ser', 'A', 24): ['ridge'], ('ser', 'A', 96): ['svr_linear', 'ridge'], ('ser', 'B', 24): ['ridge'], ('ser', 'B', 96): ['ridge'], ('ser', 'C', 24): ['xgboost', 'ridge'], ('ser', 'C', 96): ['ridge']}
FEATURE_SELECTION_MODES_TO_RUN: ['f_regression_topk', 'corr_topk']
FE

## v11 update: rescue expansion with v9-compatible candidate pools

This version is different from the previous v10 targeted-pruning notebook.

The previous targeted version reduced `candidate_models` to single-model routes. That made some routes look similar to v9, but they were not actually equivalent. For example, v9's `ser C 24h` rescue route used the candidate pool `xgboost + gbr + ridge`, while the targeted version used `xgboost` alone. That can change outer-fold model selection and OOF R².

Main changes:

- Keeps full coverage: `csf/ser × A/B/C × 24/48/72/96/120`.
- Restores v9-compatible candidate pools for critical routes.
- Expands only the high-potential areas around v9/v10 winners.
- Keeps B-family routes minimal because B is mainly for coverage.
- Uses OOF R² as the primary metric.

Critical rescue areas:

- `ser C 24h`: `xgboost + gbr + ridge`, `top_k = 10/20/30/40/50/60`, both `f_regression_topk` and `corr_topk`.
- `ser A 96h`: `svr_linear + ridge + elasticnet`, expanded top-k around 120.
- `ser A 72h`: `svr_linear + ridge + elasticnet`, expanded top-k around 120.
- `csf A 72h`: `svr_linear + ridge + gbr`, expanded top-k around 80.
- `csf A 96h`: `svr_linear + ridge + elasticnet`, expanded top-k around 40/80.
- `csf C 24/48/72/96/120`: includes `corr_topk` because v10 targeted found a strong `csf C 24h + corr_topk + top_k=20` signal.


In [5]:
# ==========================================
# Parallel config
# ==========================================
TOTAL_CPUS = os.cpu_count() or 1

# 실험 단위 병렬화 권장
# 32 CPU 기준이면 6~8 정도부터 시작하는 게 안전
N_JOBS_EXPERIMENT = 8

print("TOTAL_CPUS:", TOTAL_CPUS)
print("N_JOBS_EXPERIMENT:", N_JOBS_EXPERIMENT)

TOTAL_CPUS: 32
N_JOBS_EXPERIMENT: 8


In [6]:
def read_id_txt(path: str) -> list[str]:
    df = pd.read_csv(path)
    return df["Patient"].astype(str).tolist()


TRAIN_ID_PATH = "../../data/training_id.txt"

TRAIN_IDS = read_id_txt(TRAIN_ID_PATH)

print("n TRAIN_IDS:", len(TRAIN_IDS))


n TRAIN_IDS: 60


In [7]:
def load_target(path: Path) -> pd.Series:
    """
    patient-level target loader
    """
    y = pd.read_csv(path, index_col=0).iloc[:, 0]
    y.index = y.index.astype(str)
    y.name = "DeltaTMS"
    return y


In [8]:
# ==========================================
# Alignment helpers
# ==========================================
def align_xy(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    """
    Keep only patients that exist in both X and y.
    """
    idx = X.index.intersection(y.index)
    X = X.loc[idx].copy()
    y = y.loc[idx].copy()
    return X, y

In [9]:
def filter_sparse_features_train_test(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    min_obs_frac: float = 0.7,
    protected_keep_cols: list[str] | None = None,
):
    if protected_keep_cols is None:
        protected_keep_cols = []

    protected_keep_cols = [c for c in protected_keep_cols if c in X_train.columns]

    # 1) constant columns 제거 (train 기준)
    nunique = X_train.nunique(dropna=True)
    const_drop_cols = [
        c for c in X_train.columns
        if nunique.get(c, 0) <= 1 and c not in protected_keep_cols
    ]
    X_train_1 = X_train.drop(columns=const_drop_cols)
    X_valid_1 = X_valid.drop(columns=[c for c in const_drop_cols if c in X_valid.columns])

    # 2) missing ratio 기준 filtering (train 기준)
    obs_frac = X_train_1.notna().mean(axis=0)
    keep_cols = [
        c for c in X_train_1.columns
        if (obs_frac.get(c, 0.0) >= min_obs_frac) or (c in protected_keep_cols)
    ]

    X_train_f = X_train_1.loc[:, keep_cols].copy()
    X_valid_f = X_valid_1.reindex(columns=keep_cols).copy()

    return X_train_f, X_valid_f

In [10]:
def _median_impute_numeric_train_valid(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fit median imputation on X_train only, then apply to X_train/X_valid.
    This helper is used only for univariate scoring, not for final model fitting.
    """
    X_tr = X_train.apply(pd.to_numeric, errors="coerce")
    X_va = X_valid.apply(pd.to_numeric, errors="coerce")

    med = X_tr.median(axis=0, skipna=True)
    med = med.fillna(0.0)

    return X_tr.fillna(med), X_va.fillna(med)


def compute_univariate_feature_scores(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    protected_keep_cols: list[str] | None = None,
    min_obs: int = FEATURE_WEIGHT_MIN_OBS,
) -> pd.DataFrame:
    """
    Compute train-only univariate scores.

    Available score columns:
    - abs_corr: absolute Pearson correlation with DeltaTMS
    - f_score / p_value: sklearn f_regression score and p-value
    - neg_log10_p: larger means smaller p-value
    - hybrid_rank_score: combines corr-rank and p-value-rank

    Important:
    - This function must be called only on the training side of the current split.
    - It should never see validation/test y.
    """
    protected_keep_cols = protected_keep_cols or []
    protected_set = set(protected_keep_cols)

    candidate_cols = [c for c in X_train.columns if c not in protected_set]
    rows = []

    if len(candidate_cols) == 0:
        return pd.DataFrame(columns=[
            "feature", "abs_corr", "f_score", "p_value", "neg_log10_p",
            "corr_rank_score", "p_rank_score", "hybrid_rank_score",
            "is_protected",
        ])

    X_cand = X_train.loc[:, candidate_cols].copy()
    X_imp, _ = _median_impute_numeric_train_valid(X_cand, X_cand)
    y_num = pd.to_numeric(y_train, errors="coerce")

    valid_y = y_num.notna() & np.isfinite(y_num)
    X_imp = X_imp.loc[valid_y]
    y_num = y_num.loc[valid_y]

    if len(y_num) >= 3 and y_num.nunique(dropna=True) > 1:
        try:
            f_vals, p_vals = f_regression(X_imp, y_num)
        except Exception:
            f_vals = np.zeros(len(candidate_cols), dtype=float)
            p_vals = np.ones(len(candidate_cols), dtype=float)
    else:
        f_vals = np.zeros(len(candidate_cols), dtype=float)
        p_vals = np.ones(len(candidate_cols), dtype=float)

    for j, col in enumerate(candidate_cols):
        abs_corr = _safe_abs_corr(X_train[col], y_train, min_obs=min_obs)
        f_score = float(f_vals[j]) if np.isfinite(f_vals[j]) else 0.0
        p_value = float(p_vals[j]) if np.isfinite(p_vals[j]) else 1.0
        p_value = max(p_value, 1e-300)
        rows.append({
            "feature": col,
            "abs_corr": float(abs_corr),
            "f_score": f_score,
            "p_value": p_value,
            "neg_log10_p": float(-np.log10(p_value)),
            "is_protected": False,
        })

    score_df = pd.DataFrame(rows)

    # rank scores: higher is better
    if len(score_df) > 0:
        score_df["corr_rank_score"] = score_df["abs_corr"].rank(method="average", pct=True)
        score_df["p_rank_score"] = score_df["neg_log10_p"].rank(method="average", pct=True)
        score_df["hybrid_rank_score"] = 0.5 * score_df["corr_rank_score"] + 0.5 * score_df["p_rank_score"]

    return score_df


def select_features_train_valid(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_train: pd.Series,
    k: int | None,
    mode: str = "f_regression_topk",
    protected_keep_cols: list[str] | None = None,
    min_obs: int = FEATURE_WEIGHT_MIN_OBS,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Fold-safe feature selection.

    This replaces the old approach where target-aware top-k was applied once
    before inner CV. Here, feature selection is repeated inside each inner fold
    using only that fold's training data.

    Modes:
    - "none": keep all columns
    - "corr_topk": top-k by abs correlation with y_train
    - "f_regression_topk": top-k by univariate f_regression p-value
    - "hybrid_topk": top-k by combined corr/p-value rank
    """
    protected_keep_cols = protected_keep_cols or []
    protected_keep_cols = [c for c in protected_keep_cols if c in X_train.columns]

    if mode == "none" or k is None or k <= 0:
        selected_features = list(X_train.columns)
        score_df = pd.DataFrame({
            "feature": selected_features,
            "selected": True,
            "selection_mode": mode,
            "selection_score": np.nan,
            "is_protected": [c in protected_keep_cols for c in selected_features],
        })
        return (
            X_train.loc[:, selected_features].copy(),
            X_valid.reindex(columns=selected_features).copy(),
            score_df,
            selected_features,
        )

    score_df = compute_univariate_feature_scores(
        X_train=X_train,
        y_train=y_train,
        protected_keep_cols=protected_keep_cols,
        min_obs=min_obs,
    )

    if mode == "corr_topk":
        sort_col = "abs_corr"
    elif mode == "f_regression_topk":
        sort_col = "neg_log10_p"
    elif mode == "hybrid_topk":
        sort_col = "hybrid_rank_score"
    else:
        raise ValueError(f"Unknown feature_selection_mode: {mode}")

    candidate_cols = [c for c in X_train.columns if c not in set(protected_keep_cols)]
    k_for_candidates = max(0, int(k) - len(protected_keep_cols))

    if len(candidate_cols) == 0 or k_for_candidates == 0 or len(score_df) == 0:
        top_candidate_cols = []
    else:
        top_candidate_cols = (
            score_df.sort_values(sort_col, ascending=False)
                    .head(min(k_for_candidates, len(score_df)))["feature"]
                    .tolist()
        )

    selected_set = set(protected_keep_cols + top_candidate_cols)
    selected_features = [c for c in X_train.columns if c in selected_set]

    if len(selected_features) == 0:
        # Absolute fallback to avoid failed runs.
        selected_features = list(X_train.columns)

    out_score = score_df.copy()
    if len(out_score) > 0:
        out_score["selected"] = out_score["feature"].isin(selected_features)
        out_score["selection_mode"] = mode
        out_score["selection_score"] = out_score[sort_col]
        protected_rows = pd.DataFrame({
            "feature": protected_keep_cols,
            "abs_corr": np.nan,
            "f_score": np.nan,
            "p_value": np.nan,
            "neg_log10_p": np.nan,
            "corr_rank_score": np.nan,
            "p_rank_score": np.nan,
            "hybrid_rank_score": np.nan,
            "is_protected": True,
            "selected": True,
            "selection_mode": mode,
            "selection_score": np.nan,
        })
        out_score = pd.concat([protected_rows, out_score], ignore_index=True)

    return (
        X_train.loc[:, selected_features].copy(),
        X_valid.reindex(columns=selected_features).copy(),
        out_score,
        selected_features,
    )


def top_k_target_corr_filter(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_train: pd.Series,
    k: int | None,
    protected_keep_cols: list[str] | None = None,
    min_obs: int = FEATURE_WEIGHT_MIN_OBS,
):
    """
    Backward-compatible wrapper.
    New code should use select_features_train_valid(..., mode="corr_topk").
    """
    X_tr, X_va, _, _ = select_features_train_valid(
        X_train=X_train,
        X_valid=X_valid,
        y_train=y_train,
        k=k,
        mode="corr_topk",
        protected_keep_cols=protected_keep_cols,
        min_obs=min_obs,
    )
    return X_tr, X_va


In [11]:
from sklearn.impute import KNNImputer


def make_preprocess(X: pd.DataFrame) -> ColumnTransformer:
    cat = [c for c in X.columns if c in ["Gender", "Level"]]
    num = [c for c in X.columns if c not in cat]

    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), num),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore")),
            ]), cat),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

    try:
        preprocess.set_output(transform="pandas")
    except Exception:
        pass

    return preprocess


def make_model(model_type: str):
    """
    Model families are intentionally kept conservative because this dataset is small.
    Tree/boosting models use shallow settings through the parameter grid.
    """
    if model_type == "lasso":
        return Lasso(max_iter=50000, random_state=RANDOM_STATE, tol=1e-3)
    elif model_type == "elasticnet":
        return ElasticNet(max_iter=50000, random_state=RANDOM_STATE, tol=1e-3)
    elif model_type == "ridge":
        return Ridge()
    elif model_type == "lassolars":
        return LassoLars()
    elif model_type == "pls":
        return PLSRegression(scale=False)
    elif model_type == "svr_linear":
        return SVR(kernel="linear")
    elif model_type == "svr_rbf":
        return SVR(kernel="rbf")
    elif model_type == "knn":
        return KNeighborsRegressor()
    elif model_type == "rf":
        return RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)
    elif model_type == "extratrees":
        return ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=1)
    elif model_type == "gbr":
        return GradientBoostingRegressor(random_state=RANDOM_STATE)
    elif model_type == "xgboost":
        if not XGBOOST_AVAILABLE or XGBRegressor is None:
            raise ImportError("xgboost is not installed, so model_type='xgboost' cannot be used.")
        return XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
            verbosity=0,
        )
    elif model_type == "histgbr":
        return HistGradientBoostingRegressor(random_state=RANDOM_STATE)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")


def make_param_grid(model_type: str) -> dict:
    """
    Reduced hyperparameter grids for the training paper-style run.

    Purpose:
    - keep the full model family list
    - reduce alpha / major model grids enough to make the run practical
    - avoid the previous very long runtime from large nested CV grids
    """
    common_num_imputers = [SimpleImputer(strategy="median")]

    if model_type == "lasso":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__alpha": [0.1, 0.5],
        }

    elif model_type == "elasticnet":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__alpha": [0.05, 0.1, 0.5, 1.0],
            "model__l1_ratio": [0.1, 0.2, 0.5],
        }

    elif model_type == "ridge":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__alpha": [5.0, 10.0, 25.0, 50.0, 100.0],
        }

    elif model_type == "lassolars":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__alpha": [0.01, 0.1],
        }

    elif model_type == "pls":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_components": [2, 3],
        }

    elif model_type == "svr_linear":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__C": [0.03, 0.1, 0.3, 1.0],
            "model__epsilon": [0.1, 0.2],
        }

    elif model_type == "svr_rbf":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__C": [1.0],
            "model__epsilon": [0.1],
            "model__gamma": ["scale"],
        }

    elif model_type == "knn":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_neighbors": [3, 5],
            "model__weights": ["distance"],
        }

    elif model_type == "rf":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_estimators": [100],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [2],
            "model__max_features": ["sqrt"],
        }

    elif model_type == "extratrees":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_estimators": [100],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [2],
            "model__max_features": ["sqrt"],
        }

    elif model_type == "gbr":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_estimators": [30, 50],
            "model__learning_rate": [0.03, 0.05],
            "model__max_depth": [1],
            "model__min_samples_leaf": [3, 5],
            "model__subsample": [0.8],
        }

    elif model_type == "xgboost":
        # Conservative XGBoost grid for small-n/high-p single-omics data.
        # Goal: test regularized boosting without allowing train R2 to dominate model selection.
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_estimators": [50, 80],
            "model__max_depth": [1, 2],
            "model__learning_rate": [0.03],
            "model__subsample": [0.8],
            "model__colsample_bytree": [0.7, 0.9],
            "model__reg_alpha": [0.5, 1.0],
            "model__reg_lambda": [5.0, 10.0],
            "model__min_child_weight": [2, 4],
        }

    elif model_type == "histgbr":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__max_iter": [50],
            "model__learning_rate": [0.05],
            "model__max_leaf_nodes": [5],
            "model__l2_regularization": [0.1],
        }

    else:
        raise ValueError(f"Unknown model_type: {model_type}")

def adjust_param_grid_for_training_data(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    model_type: str,
) -> list[dict]:
    """
    Remove parameter settings that cannot work for the current train fold.
    Example: PLS n_components must be <= min(n_samples - 1, n_features).
    """
    grid = list(ParameterGrid(make_param_grid(model_type=model_type)))

    n_samples = int(X_train.shape[0])
    n_features = int(X_train.shape[1])

    valid_grid = []
    for params in grid:
        ok = True

        if model_type == "pls":
            n_comp = int(params.get("model__n_components", 1))
            if n_comp > max(1, min(n_samples - 1, n_features)):
                ok = False

        if model_type == "knn":
            n_neighbors = int(params.get("model__n_neighbors", 5))
            if n_neighbors >= n_samples:
                ok = False

        if ok:
            valid_grid.append(params)

    if len(valid_grid) == 0:
        # Fallback to a safe default for very small folds.
        if model_type == "pls":
            return [{
                "preprocess__num__imputer": SimpleImputer(strategy="median"),
                "model__n_components": 1,
            }]
        if model_type == "knn":
            return [{
                "preprocess__num__imputer": SimpleImputer(strategy="median"),
                "model__n_neighbors": max(1, min(3, n_samples - 1)),
                "model__weights": "distance",
            }]
        return grid

    return valid_grid


def make_pipeline(
    X: pd.DataFrame,
    model_type: str,
    feature_weight_map: dict[str, float] | None = None,
) -> Pipeline:
    preprocess = make_preprocess(X)
    model = make_model(model_type)

    steps = [
        ("preprocess", preprocess),
    ]

    if USE_FEATURE_WEIGHTING:
        steps.append((
            "feature_weight",
            FeatureWeightTransformer(
                feature_weight_map=feature_weight_map,
                default_weight=FEATURE_WEIGHT_DEFAULT,
                enabled=USE_FEATURE_WEIGHTING,
            ),
        ))

    steps.append(("model", model))

    return Pipeline(steps)


def predict_1d(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    """
    Some models, e.g. PLSRegression, return shape (n, 1).
    Standardize all predictions to 1D.
    """
    return np.ravel(model.predict(X))


In [12]:
def build_feature_path(feature_mode: str, tissue: str, combo: str, tp: int) -> Path:
    if feature_mode == "omics":
        return TP_DIR / f"x_omics_{tissue}_{combo}_{tp}.csv"
    elif feature_mode == "omicsdelta":
        return TP_DIR / f"x_omicsdelta_{tissue}_{combo}_{tp}.csv"
    else:
        raise ValueError(f"Unknown feature_mode: {feature_mode}")

def build_clinical_feature_path(tissue: str, combo: str, tp: int) -> Path:
    return TP_DIR / f"x_clinical_{tissue}_{combo}_{tp}.csv"


def load_feature_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "Patient" not in df.columns:
        raise ValueError(f"'Patient' not found in {path}")
    df = df.set_index("Patient")
    df.index = df.index.astype(str)
    return df

def dataframe_content_signature(df: pd.DataFrame) -> str:
    """
    Lightweight diagnostic hash used to check whether two TP feature matrices
    are accidentally identical. This is not used for modeling.
    """
    if df is None or len(df) == 0:
        return "EMPTY"
    tmp = df.copy()
    tmp = tmp.sort_index(axis=0).sort_index(axis=1)
    col_part = "|".join(map(str, tmp.columns))
    shape_part = f"{tmp.shape[0]}x{tmp.shape[1]}"
    # hash_pandas_object handles mixed dtypes after string fallback; keep it lightweight.
    try:
        values_hash = pd.util.hash_pandas_object(tmp.astype(str), index=True).values.tobytes()
    except Exception:
        values_hash = tmp.to_csv(index=True).encode("utf-8")
    return hashlib.md5((shape_part + "|" + col_part).encode("utf-8") + values_hash).hexdigest()


In [13]:
def get_clinical_columns() -> list[str]:
    return ["Age", "Gender", "Level"]


def get_hard_forbidden_proxy_columns() -> list[str]:
    return [
        "TMS_Base", "UEMS_Base", "LEMS_Base",
        "TMS_6mo", "UEMS_6mo", "LEMS_6mo",
        "AIS_6mo",
        "DeltaTMS", "DeltaUEMS", "DeltaLEMS",
    ]


def assert_no_forbidden_proxy_columns(
    X: pd.DataFrame,
    use_light_clinical: bool = False,
):
    hard_bad = [c for c in X.columns if c in get_hard_forbidden_proxy_columns()]
    if len(hard_bad) > 0:
        raise ValueError(f"Hard forbidden proxy/leakage columns found: {hard_bad}")

    all_clinical_like = [
        "Age", "Gender", "Level", "AIS", "AIS_Base", "AIS_Base_Numeric",
        "TMS_Base", "UEMS_Base", "LEMS_Base",
        "TMS_6mo", "UEMS_6mo", "LEMS_6mo", "AIS_6mo",
        "DeltaTMS", "DeltaUEMS", "DeltaLEMS",
    ]

    allowed = set(get_clinical_columns()) if use_light_clinical else set()
    soft_bad = [
        c for c in X.columns
        if (c in all_clinical_like and c not in allowed and c not in get_hard_forbidden_proxy_columns())
    ]

    if len(soft_bad) > 0:
        raise ValueError(f"Unexpected clinical columns found for this mode: {soft_bad}")

In [14]:
def load_light_clinical_block(
    tissue: str,
    combo: str,
    tp: int,
) -> pd.DataFrame:
    clin_path = build_clinical_feature_path(tissue=tissue, combo=combo, tp=tp)
    clin_df = load_feature_csv(clin_path)

    keep_cols = [c for c in get_clinical_columns() if c in clin_df.columns]
    clin_df = clin_df.loc[:, keep_cols].copy()

    missing_cols = [c for c in get_clinical_columns() if c not in clin_df.columns]
    if len(missing_cols) > 0:
        print(f"[WARN] missing light clinical columns in {clin_path.name}: {missing_cols}")

    if "Age" in clin_df.columns:
        clin_df["Age"] = pd.to_numeric(clin_df["Age"], errors="coerce")

    return clin_df


def merge_with_light_clinical(
    X_feat: pd.DataFrame,
    tissue: str,
    combo: str,
    tp: int,
) -> pd.DataFrame:
    clin_df = load_light_clinical_block(tissue=tissue, combo=combo, tp=tp)

    common_ids = X_feat.index.intersection(clin_df.index)
    X_feat2 = X_feat.loc[common_ids].copy()
    clin_df2 = clin_df.loc[common_ids].copy()

    X_merged = pd.concat([X_feat2, clin_df2], axis=1)

    dup_cols = X_merged.columns[X_merged.columns.duplicated()].tolist()
    if len(dup_cols) > 0:
        raise ValueError(f"Duplicate columns after light clinical merge: {dup_cols}")

    return X_merged


In [15]:
def encode_clinical_train_valid(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Clinical block은 train 기준으로만 encoding한다.
    반환:
      - encoded X_train
      - encoded X_valid
      - encoded clinical column names
    """
    base_clin_cols = [c for c in get_clinical_columns() if c in X_train.columns]
    if len(base_clin_cols) == 0:
        return X_train, X_valid, []

    tr_clin = X_train.loc[:, base_clin_cols].copy()
    va_clin = X_valid.loc[:, [c for c in base_clin_cols if c in X_valid.columns]].copy()

    tr_other = X_train.drop(columns=base_clin_cols, errors="ignore").copy()
    va_other = X_valid.drop(columns=base_clin_cols, errors="ignore").copy()

    if "Age" in tr_clin.columns:
        tr_clin["Age"] = pd.to_numeric(tr_clin["Age"], errors="coerce")
    if "Age" in va_clin.columns:
        va_clin["Age"] = pd.to_numeric(va_clin["Age"], errors="coerce")

    cat_cols = [c for c in ["Gender", "Level"] if c in tr_clin.columns]

    if len(cat_cols) > 0:
        tr_clin_enc = pd.get_dummies(
            tr_clin,
            columns=cat_cols,
            dummy_na=True,
            dtype=float,
        )
        va_clin_enc = pd.get_dummies(
            va_clin,
            columns=cat_cols,
            dummy_na=True,
            dtype=float,
        )
        va_clin_enc = va_clin_enc.reindex(columns=tr_clin_enc.columns, fill_value=0.0)
    else:
        tr_clin_enc = tr_clin.copy()
        va_clin_enc = va_clin.copy()

    tr_clin_enc = tr_clin_enc.apply(pd.to_numeric, errors="coerce")
    va_clin_enc = va_clin_enc.apply(pd.to_numeric, errors="coerce")

    encoded_clin_cols = tr_clin_enc.columns.tolist()

    X_train_out = pd.concat([tr_other, tr_clin_enc], axis=1)
    X_valid_out = pd.concat([va_other, va_clin_enc], axis=1)

    return X_train_out, X_valid_out, encoded_clin_cols


In [16]:
# # ==========================================
# # Inner CV tuning
# # ==========================================
# def fit_best_model(
#     X_train: pd.DataFrame,
#     y_train: pd.Series,
#     groups_train: np.ndarray,
#     model_type: str,
#     n_splits_inner: int = 3,
# ):
#     """
#     Tune hyperparameters only on the training fold.
#     """
#     pipe = make_pipeline(X_train, model_type=model_type)
#     param_grid = make_param_grid(model_type=model_type)

#     if model_type in ["lasso", "elasticnet"] and "select__k" in param_grid:
#         preprocess = make_preprocess(X_train)
#         X_train_tx = preprocess.fit_transform(X_train, y_train)
#         n_features_after_preprocess = X_train_tx.shape[1]

#         valid_k = sorted({
#             int(k)
#             for k in param_grid["select__k"]
#             if int(k) <= n_features_after_preprocess
#         })

#         if len(valid_k) == 0:
#             valid_k = [max(1, n_features_after_preprocess)]

#         param_grid["select__k"] = valid_k

#     inner_cv = GroupKFold(n_splits=n_splits_inner)

#     gs = GridSearchCV(
#         estimator=pipe,
#         param_grid=param_grid,
#         scoring="r2",
#         cv=inner_cv,
#         n_jobs=12,
#         refit=True,
#     )
#     gs.fit(X_train, y_train, groups=groups_train)

#     return gs.best_estimator_, gs.best_params_, gs.best_score_

In [17]:
class FeatureWeightTransformer(BaseEstimator, TransformerMixin):
    """
    Apply precomputed feature-wise weights after preprocessing/scaling.

    Why after preprocessing?
    - If weights are applied before StandardScaler, StandardScaler can cancel out most
      multiplicative effects.
    - Applying after preprocessing means the model sees the weighted feature matrix.

    Leakage rule:
    - feature_weight_map must be computed outside this transformer using train data only.
    - validation/test y is never used here.
    """
    def __init__(
        self,
        feature_weight_map: dict[str, float] | None = None,
        default_weight: float = 1.0,
        enabled: bool = True,
    ):
        self.feature_weight_map = feature_weight_map
        self.default_weight = default_weight
        self.enabled = enabled

    def fit(self, X, y=None):
        if not self.enabled or self.feature_weight_map is None:
            self.weights_ = np.ones(X.shape[1], dtype=float)
            self.feature_names_ = list(getattr(X, "columns", [f"x{i}" for i in range(X.shape[1])]))
            return self

        if hasattr(X, "columns"):
            self.feature_names_ = list(X.columns)
            weights = []
            for feat in self.feature_names_:
                feat_str = str(feat)

                # exact match for numeric/transformed numeric columns
                w = self.feature_weight_map.get(feat_str, None)

                # fallback for one-hot names such as Gender_M or Level_C
                if w is None:
                    base = feat_str.split("_")[0]
                    w = self.feature_weight_map.get(base, self.default_weight)

                weights.append(float(w))

            self.weights_ = np.asarray(weights, dtype=float)
        else:
            # fallback: if sklearn cannot output pandas, keep neutral weighting
            self.feature_names_ = [f"x{i}" for i in range(X.shape[1])]
            self.weights_ = np.ones(X.shape[1], dtype=float)

        return self

    def transform(self, X):
        if not self.enabled:
            return X

        if hasattr(X, "copy") and hasattr(X, "columns"):
            Xw = X.copy()
            for col, w in zip(Xw.columns, self.weights_):
                Xw[col] = Xw[col] * float(w)
            return Xw

        return X * self.weights_


def _safe_abs_corr(x: pd.Series, y: pd.Series, min_obs: int) -> float:
    x_num = pd.to_numeric(x, errors="coerce")
    y_num = pd.to_numeric(y, errors="coerce")
    mask = x_num.notna() & y_num.notna() & np.isfinite(x_num) & np.isfinite(y_num)

    if int(mask.sum()) < min_obs:
        return 0.0

    xv = x_num.loc[mask]
    yv = y_num.loc[mask]

    if xv.nunique(dropna=True) <= 1 or yv.nunique(dropna=True) <= 1:
        return 0.0

    corr = xv.corr(yv)
    if pd.isna(corr) or not np.isfinite(corr):
        return 0.0

    return float(abs(corr))


def compute_feature_weight_map_from_train(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    protected_keep_cols: list[str] | None = None,
    min_weight: float = FEATURE_WEIGHT_MIN,
    max_weight: float = FEATURE_WEIGHT_MAX,
    min_obs: int = FEATURE_WEIGHT_MIN_OBS,
) -> tuple[dict[str, float], pd.DataFrame]:
    """
    Compute feature weights using outer-train data only.

    Current score:
    - absolute Pearson correlation between feature and DeltaTMS within train data
    - clinical/protected columns are kept at weight 1.0 by default

    This is intentionally simple and conservative. It can be replaced later by
    selection-frequency or coefficient-stability scoring, but this version is safer
    as the first feature-weighting experiment.
    """
    protected_keep_cols = protected_keep_cols or []
    protected_set = set(protected_keep_cols)

    rows = []
    for col in X_train.columns:
        is_protected = col in protected_set

        if FEATURE_WEIGHT_EXCLUDE_CLINICAL and is_protected:
            score = 0.0
            weight = FEATURE_WEIGHT_DEFAULT
        else:
            score = _safe_abs_corr(X_train[col], y_train, min_obs=min_obs)
            weight = np.nan

        rows.append({
            "feature": col,
            "feature_score": float(score),
            "is_protected": bool(is_protected),
            "weight": weight,
        })

    score_df = pd.DataFrame(rows)

    eligible = score_df["weight"].isna()
    if eligible.any():
        s = score_df.loc[eligible, "feature_score"].astype(float)
        s_min = float(s.min())
        s_max = float(s.max())

        if np.isclose(s_min, s_max):
            scaled = pd.Series(0.0, index=s.index)
        else:
            scaled = (s - s_min) / (s_max - s_min)

        score_df.loc[eligible, "weight"] = min_weight + scaled * (max_weight - min_weight)

    score_df["weight"] = score_df["weight"].fillna(FEATURE_WEIGHT_DEFAULT).astype(float)
    score_df = score_df.sort_values(
        ["weight", "feature_score"],
        ascending=[False, False],
    ).reset_index(drop=True)

    weight_map = dict(zip(score_df["feature"].astype(str), score_df["weight"].astype(float)))

    return weight_map, score_df


def build_feature_weights(
    feature_score_df: pd.DataFrame,
    mode: str = "soft",
    top_k: int = 50,
    high_weight: float = 1.5,
    low_weight: float = 0.5,
    min_weight: float = 0.5,
    max_weight: float = 1.5,
):
    """
    Backward-compatible helper.
    Existing code may still call this function.
    """
    df = feature_score_df.copy()

    if len(df) == 0:
        return pd.DataFrame(columns=["feature", "weight"])

    if mode == "topk":
        df["weight"] = low_weight
        top_feats = df.head(top_k)["feature"].tolist()
        df.loc[df["feature"].isin(top_feats), "weight"] = high_weight

    elif mode == "soft":
        s = df["feature_score"].values.astype(float)
        s_min, s_max = s.min(), s.max()
        if np.isclose(s_min, s_max):
            scaled = np.ones_like(s)
        else:
            scaled = (s - s_min) / (s_max - s_min)
        df["weight"] = min_weight + scaled * (max_weight - min_weight)

    else:
        raise ValueError(f"Unknown weighting mode: {mode}")

    return df[["feature", "weight"]].copy()


In [18]:
def extract_final_feature_info(fitted_pipeline: Pipeline) -> pd.DataFrame:
    preprocess = fitted_pipeline.named_steps["preprocess"]
    feature_names = list(preprocess.get_feature_names_out())

    model = fitted_pipeline.named_steps["model"]

    coef = None
    if hasattr(model, "coef_"):
        coef = np.ravel(model.coef_)
    elif hasattr(model, "x_weights_"):
        # PLS does not expose coef_ in exactly the same way across sklearn versions.
        # x_weights_ is not identical to regression coefficients, but it is useful as a trace.
        coef = np.ravel(getattr(model, "x_weights_", np.array([])))

    if coef is not None and len(coef) == len(feature_names):
        coef_list = coef.tolist()
    else:
        coef_list = [np.nan] * len(feature_names)

    feature_df = pd.DataFrame({
        "feature": feature_names,
        "coef": coef_list,
        "abs_coef": np.abs(coef_list),
    })

    if USE_FEATURE_WEIGHTING and "feature_weight" in fitted_pipeline.named_steps:
        fw = fitted_pipeline.named_steps["feature_weight"]
        weight_by_processed = dict(zip(getattr(fw, "feature_names_", []), getattr(fw, "weights_", [])))
        feature_df["feature_weight"] = feature_df["feature"].map(weight_by_processed).fillna(FEATURE_WEIGHT_DEFAULT)
    else:
        feature_df["feature_weight"] = FEATURE_WEIGHT_DEFAULT

    feature_df = feature_df.sort_values(
        "abs_coef",
        ascending=False,
        na_position="last",
    ).reset_index(drop=True)

    return feature_df


def make_regression_strata(
    y: pd.Series,
    n_bins: int,
    n_splits: int,
) -> pd.Series:
    """
    Quantile-based strata for regression CV.
    """
    y_num = pd.to_numeric(y, errors="coerce")

    if y_num.notna().sum() < n_splits:
        return pd.Series(index=y.index, data=0).astype(int)

    max_bins = min(n_bins, int(y_num.nunique()))

    for bins in range(max_bins, 1, -1):
        try:
            y_bin = pd.qcut(
                y_num,
                q=bins,
                labels=False,
                duplicates="drop",
            )
            y_bin = pd.Series(y_bin, index=y.index)

            counts = y_bin.value_counts(dropna=True)

            if len(counts) >= 2 and counts.min() >= n_splits:
                return y_bin.fillna(-1).astype(int)

        except Exception:
            continue

    return pd.Series(index=y.index, data=0).astype(int)


def make_outer_cv_splits(
    X: pd.DataFrame,
    y: pd.Series,
    groups: np.ndarray,
    n_splits: int,
):
    """
    Outer CV split helper.
    """
    if USE_STRATIFIED_OUTER_CV and X.index.is_unique:
        y_bins = make_regression_strata(
            y=y,
            n_bins=N_Y_BINS_FOR_OUTER_CV,
            n_splits=n_splits,
        )

        if y_bins.nunique(dropna=True) >= 2:
            outer_cv = StratifiedKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=OUTER_CV_RANDOM_STATE,
            )
            return list(outer_cv.split(X, y_bins)), "StratifiedKFold_y_bins"

    outer_cv = GroupKFold(n_splits=n_splits)
    return list(outer_cv.split(X, y, groups=groups)), "GroupKFold_fallback"


def make_inner_cv_splits(
    X: pd.DataFrame,
    y: pd.Series,
    groups: np.ndarray,
    n_splits: int,
):
    """
    Inner CV split helper.

    목적:
    - model/alpha/top-k 선택이 특정 fold의 y 분포 때문에 흔들리는 것을 줄인다.
    - patient index가 unique이면 y-bin stratified split을 사용한다.
    - 실패하면 GroupKFold로 fallback한다.
    """
    if X.index.is_unique:
        y_bins = make_regression_strata(
            y=y,
            n_bins=N_Y_BINS_FOR_OUTER_CV,
            n_splits=n_splits,
        )

        if y_bins.nunique(dropna=True) >= 2:
            inner_cv = StratifiedKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=RANDOM_STATE,
            )
            return list(inner_cv.split(X, y_bins)), "StratifiedKFold_y_bins"

    inner_cv = GroupKFold(n_splits=n_splits)
    return list(inner_cv.split(X, y, groups=groups)), "GroupKFold_fallback"


def fit_best_model_only(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    candidate_models: list[str],
    n_splits_inner: int = 3,
    protected_keep_cols: list[str] | None = None,
    feature_selection_mode: str = "f_regression_topk",
    top_k_after_sparse: int | None = 100,
):
    """
    Select model/hyperparameters using train-side inner OOF R².

    Paper-style leakage rule:
    - In each inner fold, feature selection is fitted only on inner-train.
    - Inner-validation y is never used for feature selection or feature weighting.
    - Final feature selection is fitted only on the full predefined training set.
    - Holdout test y is never used until final evaluation.
    """
    protected_keep_cols = protected_keep_cols or []

    search_rows = []
    best_obj = None
    best_score = -np.inf

    inner_splits, inner_cv_type = make_inner_cv_splits(
        X=X_train,
        y=y_train,
        groups=groups_train,
        n_splits=n_splits_inner,
    )

    for model_type in candidate_models:
        grid = adjust_param_grid_for_training_data(
            X_train=X_train,
            y_train=y_train,
            model_type=model_type,
        )

        for params in grid:
            inner_oof = pd.Series(index=X_train.index, dtype=float)
            inner_fold_scores = []
            inner_n_features = []

            for inner_fold, (tr_idx, va_idx) in enumerate(inner_splits, start=1):
                X_tr_raw = X_train.iloc[tr_idx].copy()
                y_tr = y_train.iloc[tr_idx].copy()
                X_va_raw = X_train.iloc[va_idx].copy()
                y_va = y_train.iloc[va_idx].copy()

                X_tr, X_va, _, selected_features = select_features_train_valid(
                    X_train=X_tr_raw,
                    X_valid=X_va_raw,
                    y_train=y_tr,
                    k=top_k_after_sparse,
                    mode=feature_selection_mode,
                    protected_keep_cols=protected_keep_cols,
                )
                inner_n_features.append(len(selected_features))

                if USE_FEATURE_WEIGHTING:
                    inner_weight_map, _ = compute_feature_weight_map_from_train(
                        X_train=X_tr,
                        y_train=y_tr,
                        protected_keep_cols=[c for c in protected_keep_cols if c in X_tr.columns],
                    )
                else:
                    inner_weight_map = None

                pipe = make_pipeline(
                    X_tr,
                    model_type=model_type,
                    feature_weight_map=inner_weight_map,
                )
                pipe.set_params(**params)
                pipe.fit(X_tr, y_tr)

                pred = predict_1d(pipe, X_va)
                inner_oof.iloc[va_idx] = pred

                try:
                    inner_fold_scores.append(float(r2_score(y_va, pred)))
                except Exception:
                    inner_fold_scores.append(np.nan)

            valid_mask = inner_oof.notna()

            if valid_mask.sum() >= 2:
                inner_oof_r2 = float(
                    r2_score(
                        y_train.loc[valid_mask],
                        inner_oof.loc[valid_mask],
                    )
                )
                inner_oof_mae = float(
                    mean_absolute_error(
                        y_train.loc[valid_mask],
                        inner_oof.loc[valid_mask],
                    )
                )
            else:
                inner_oof_r2 = np.nan
                inner_oof_mae = np.nan

            inner_mean_fold_r2 = (
                float(np.nanmean(inner_fold_scores))
                if len(inner_fold_scores) > 0 and not np.all(pd.isna(inner_fold_scores))
                else np.nan
            )

            inner_std_fold_r2 = (
                float(np.nanstd(inner_fold_scores))
                if len(inner_fold_scores) > 0 and not np.all(pd.isna(inner_fold_scores))
                else np.nan
            )

            if pd.notna(inner_oof_r2):
                selection_score = float(inner_oof_r2)
                if pd.notna(inner_std_fold_r2):
                    selection_score -= INNER_SELECTION_STD_PENALTY * float(inner_std_fold_r2)
            else:
                selection_score = np.nan

            search_rows.append({
                "model_type": model_type,
                "params": json.dumps(params, default=str),
                "inner_mean_r2": inner_oof_r2,
                "inner_oof_r2": inner_oof_r2,
                "inner_oof_mae": inner_oof_mae,
                "inner_mean_fold_r2": inner_mean_fold_r2,
                "inner_std_fold_r2": inner_std_fold_r2,
                "selection_score_stability_penalized": selection_score,
                "inner_cv_type": inner_cv_type,
                "feature_selection_mode": feature_selection_mode,
                "top_k_after_sparse": top_k_after_sparse,
                "mean_inner_selected_features": (
                    float(np.mean(inner_n_features)) if len(inner_n_features) > 0 else np.nan
                ),
                "min_inner_selected_features": (
                    int(np.min(inner_n_features)) if len(inner_n_features) > 0 else np.nan
                ),
                "max_inner_selected_features": (
                    int(np.max(inner_n_features)) if len(inner_n_features) > 0 else np.nan
                ),
                "use_feature_weighting": USE_FEATURE_WEIGHTING,
                "feature_weight_min": FEATURE_WEIGHT_MIN if USE_FEATURE_WEIGHTING else np.nan,
                "feature_weight_max": FEATURE_WEIGHT_MAX if USE_FEATURE_WEIGHTING else np.nan,
            })

            if pd.notna(selection_score) and selection_score > best_score:
                best_score = selection_score
                best_obj = {
                    "model_type": model_type,
                    "params": params,
                }

    if best_obj is None:
        raise ValueError("No valid model/parameter combination found in fit_best_model_only")

    # Final train-only feature selection for the holdout test model.
    X_train_sel, _, final_selection_df, selected_features = select_features_train_valid(
        X_train=X_train,
        X_valid=X_train,
        y_train=y_train,
        k=top_k_after_sparse,
        mode=feature_selection_mode,
        protected_keep_cols=protected_keep_cols,
    )

    if USE_FEATURE_WEIGHTING:
        final_weight_map, final_weight_df = compute_feature_weight_map_from_train(
            X_train=X_train_sel,
            y_train=y_train,
            protected_keep_cols=[c for c in protected_keep_cols if c in X_train_sel.columns],
        )
    else:
        final_weight_map = None
        final_weight_df = pd.DataFrame(columns=["feature", "feature_score", "is_protected", "weight"])

    final_pipe = make_pipeline(
        X_train_sel,
        model_type=best_obj["model_type"],
        feature_weight_map=final_weight_map,
    )
    final_pipe.set_params(**best_obj["params"])
    final_pipe.fit(X_train_sel, y_train)

    search_df = pd.DataFrame(search_rows)
    if len(search_df) > 0:
        search_df = search_df.sort_values(
            "selection_score_stability_penalized",
            ascending=False,
            na_position="last",
        ).reset_index(drop=True)

    return {
        "best_model_type": best_obj["model_type"],
        "best_params": best_obj["params"],
        "best_score": best_score,
        "search_df": search_df,
        "fitted_pipeline": final_pipe,
        "feature_weight_df": final_weight_df,
        "feature_selection_df": final_selection_df,
        "selected_features": selected_features,
    }


def get_inner_oof_predictions_model_only(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    candidate_models: list[str],
    n_splits_inner: int = 3,
    protected_keep_cols: list[str] | None = None,
    feature_selection_mode: str = "f_regression_topk",
    top_k_after_sparse: int | None = 100,
):
    """
    Diagnostic train-inner OOF predictions using the same paper-style fold-safe selection.
    """
    inner_splits, inner_cv_type = make_inner_cv_splits(
        X=X_train,
        y=y_train,
        groups=groups_train,
        n_splits=n_splits_inner,
    )

    oof = pd.Series(index=X_train.index, dtype=float)

    for tr_idx, va_idx in inner_splits:
        X_tr = X_train.iloc[tr_idx].copy()
        y_tr = y_train.iloc[tr_idx].copy()
        X_va = X_train.iloc[va_idx].copy()

        fit_result = fit_best_model_only(
            X_train=X_tr,
            y_train=y_tr,
            groups_train=groups_train[tr_idx],
            candidate_models=candidate_models,
            n_splits_inner=n_splits_inner,
            protected_keep_cols=protected_keep_cols,
            feature_selection_mode=feature_selection_mode,
            top_k_after_sparse=top_k_after_sparse,
        )

        selected_features = fit_result["selected_features"]
        X_va_sel = X_va.reindex(columns=selected_features).copy()

        best_pipe = fit_result["fitted_pipeline"]
        pred_va = predict_1d(best_pipe, X_va_sel)

        oof.iloc[va_idx] = pred_va

    return oof


In [19]:
def generate_weight_grid(n_models: int, step: float = 0.1):
    """
    Generate nonnegative weight tuples that sum to 1.
    Example:
      n_models=2, step=0.5 -> [(0.0,1.0),(0.5,0.5),(1.0,0.0)]
    """
    if n_models < 1:
        raise ValueError("n_models must be >= 1")
    if step <= 0 or step > 1:
        raise ValueError("step must be in (0, 1]")

    step_int = int(round(1 / step))
    if not np.isclose(step_int * step, 1.0):
        raise ValueError("step must divide 1.0 exactly, e.g. 0.5, 0.25, 0.2, 0.1")

    grids = []

    def backtrack(prefix, remaining, depth):
        if depth == n_models - 1:
            grids.append(tuple(prefix + [remaining / step_int]))
            return
        for v in range(remaining + 1):
            backtrack(prefix + [v / step_int], remaining - v, depth + 1)

    backtrack([], step_int, 0)
    return grids

In [20]:
def encode_clinical_train_valid(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Clinical을 항상 fold 내부에서 encoding한다.
    반환:
      - encoded X_train
      - encoded X_valid
      - encoded clinical column names
    """
    base_clin_cols = [c for c in get_clinical_columns() if c in X_train.columns]
    if len(base_clin_cols) == 0:
        return X_train, X_valid, []

    tr_clin = X_train.loc[:, base_clin_cols].copy()
    va_clin = X_valid.loc[:, [c for c in base_clin_cols if c in X_valid.columns]].copy()

    tr_other = X_train.drop(columns=base_clin_cols, errors="ignore").copy()
    va_other = X_valid.drop(columns=base_clin_cols, errors="ignore").copy()

    if "Age" in tr_clin.columns:
        tr_clin["Age"] = pd.to_numeric(tr_clin["Age"], errors="coerce")
    if "Age" in va_clin.columns:
        va_clin["Age"] = pd.to_numeric(va_clin["Age"], errors="coerce")

    cat_cols = [c for c in ["Gender", "Level"] if c in tr_clin.columns]

    if len(cat_cols) > 0:
        tr_clin_enc = pd.get_dummies(
            tr_clin,
            columns=cat_cols,
            dummy_na=True,
            dtype=float,
        )
        va_clin_enc = pd.get_dummies(
            va_clin,
            columns=cat_cols,
            dummy_na=True,
            dtype=float,
        )
        va_clin_enc = va_clin_enc.reindex(columns=tr_clin_enc.columns, fill_value=0.0)
    else:
        tr_clin_enc = tr_clin.copy()
        va_clin_enc = va_clin.copy()

    tr_clin_enc = tr_clin_enc.apply(pd.to_numeric, errors="coerce")
    va_clin_enc = va_clin_enc.apply(pd.to_numeric, errors="coerce")

    encoded_clin_cols = tr_clin_enc.columns.tolist()

    X_train_out = pd.concat([tr_other, tr_clin_enc], axis=1)
    X_valid_out = pd.concat([va_other, va_clin_enc], axis=1)

    return X_train_out, X_valid_out, encoded_clin_cols

In [21]:
def run_single_omics_train_oof_cv(
    X: pd.DataFrame,
    y: pd.Series,
    tissue: str,
    combo: str,
    tp: int,
    train_ids: list[str],
    candidate_models: list[str],
    feature_mode: str,
    clinical_mode: str = "light",
    feature_selection_mode: str = "f_regression_topk",
    min_obs_frac: float = MIN_OBS_FRAC,
    top_k_after_sparse: int | None = 100,
):
    """
    TRAIN-ONLY paper-style single-omics evaluation.

    Main metric:
    - outer-CV OOF R² computed only on TRAIN_IDS

    Leakage control:
    - outer-valid y is never used for sparse filtering, feature selection, feature weighting, or tuning
    - inner feature selection is fitted only on inner-train folds
    - all OOF predictions are generated from models that did not train on that patient
    """
    train_ids_in = [pid for pid in train_ids if pid in X.index]
    X_all = X.loc[train_ids_in].copy()
    y_all_train = y.loc[train_ids_in].copy()

    print("\n[TRAIN-ONLY OOF CV CHECK]")
    print("clinical_mode:", clinical_mode)
    print("feature_selection_mode:", feature_selection_mode)
    print("top_k_after_sparse:", top_k_after_sparse)
    print("X_all shape:", X_all.shape)
    print("y_all_train shape:", y_all_train.shape)
    print("n unique train ids:", X_all.index.nunique())

    delta_cols = [c for c in X_all.columns if "_d" in str(c)]
    clin_cols_present = [c for c in get_clinical_columns() if c in X_all.columns]
    print("delta cols count:", len(delta_cols))
    print("clinical cols present:", clin_cols_present)

    if len(X_all) < N_SPLITS_OUTER:
        raise ValueError("not enough training samples for outer CV")

    outer_splits, outer_cv_type = make_outer_cv_splits(
        X=X_all,
        y=y_all_train,
        groups=X_all.index.to_numpy(),
        n_splits=N_SPLITS_OUTER,
    )

    print("outer_cv_type:", outer_cv_type)
    print("n_outer_folds:", len(outer_splits))

    oof_pred = pd.Series(index=X_all.index, dtype=float)
    fold_metric_rows = []
    oof_rows = []
    search_dfs = []
    feature_dfs = []
    feature_weight_dfs = []
    feature_selection_dfs = []

    protected_base_cols = get_clinical_columns() if clinical_mode == "light" else []

    for outer_fold, (tr_idx, va_idx) in enumerate(outer_splits, start=1):
        X_tr_raw = X_all.iloc[tr_idx].copy()
        y_tr = y_all_train.iloc[tr_idx].copy()
        X_va_raw = X_all.iloc[va_idx].copy()
        y_va = y_all_train.iloc[va_idx].copy()

        print(f"\n[OUTER FOLD {outer_fold}]")
        print("raw train shape:", X_tr_raw.shape, "| valid shape:", X_va_raw.shape)

        X_tr_f, X_va_f = filter_sparse_features_train_test(
            X_train=X_tr_raw,
            X_valid=X_va_raw,
            min_obs_frac=min_obs_frac,
            protected_keep_cols=protected_base_cols,
        )

        n_before_sparse = X_tr_raw.shape[1]
        n_after_sparse = X_tr_f.shape[1]

        if X_tr_f.shape[1] == 0:
            raise ValueError(f"outer_fold={outer_fold}: no features left after sparse filtering")

        if clinical_mode == "light":
            X_tr_f, X_va_f, encoded_clin_cols = encode_clinical_train_valid(
                X_train=X_tr_f,
                X_valid=X_va_f,
            )
            protected_keep_cols = encoded_clin_cols
        else:
            encoded_clin_cols = []
            protected_keep_cols = []

        non_numeric_cols = X_tr_f.select_dtypes(exclude=[np.number, "bool"]).columns.tolist()
        if len(non_numeric_cols) > 0:
            raise TypeError(f"outer_fold={outer_fold}: non-numeric columns remain: {non_numeric_cols[:20]}")

        fit_result = fit_best_model_only(
            X_train=X_tr_f,
            y_train=y_tr,
            groups_train=X_tr_f.index.to_numpy(),
            candidate_models=candidate_models,
            n_splits_inner=N_SPLITS_INNER,
            protected_keep_cols=protected_keep_cols,
            feature_selection_mode=feature_selection_mode,
            top_k_after_sparse=top_k_after_sparse,
        )

        selected_features = fit_result["selected_features"]
        X_tr_model = X_tr_f.loc[:, selected_features].copy()
        X_va_model = X_va_f.reindex(columns=selected_features).copy()

        best_pipe = fit_result["fitted_pipeline"]

        pred_tr = predict_1d(best_pipe, X_tr_model)
        pred_va = predict_1d(best_pipe, X_va_model)

        oof_pred.iloc[va_idx] = pred_va

        train_r2 = float(r2_score(y_tr, pred_tr)) if len(y_tr) >= 2 else np.nan
        valid_r2 = float(r2_score(y_va, pred_va)) if len(y_va) >= 2 else np.nan
        train_mae = float(mean_absolute_error(y_tr, pred_tr))
        valid_mae = float(mean_absolute_error(y_va, pred_va))

        print(
            f"fold={outer_fold} | best={fit_result['best_model_type']} | "
            f"valid_r2={valid_r2:.4f} | train_r2={train_r2:.4f} | "
            f"n_features={len(selected_features)}"
        )

        fold_metric_rows.append({
            "outer_fold": outer_fold,
            "outer_cv_type": outer_cv_type,
            "clinical_mode": clinical_mode,
            "feature_selection_mode": feature_selection_mode,
            "tissue": tissue,
            "combo": combo,
            "tp": tp,
            "best_model_type": fit_result["best_model_type"],
            "best_params": json.dumps(fit_result["best_params"], default=str),
            "best_inner_oof_r2": fit_result["best_score"],
            "train_r2": train_r2,
            "valid_r2": valid_r2,
            "train_mae": train_mae,
            "valid_mae": valid_mae,
            "n_train": len(y_tr),
            "n_valid": len(y_va),
            "n_features_before_sparse": n_before_sparse,
            "n_features_after_sparse": n_after_sparse,
            "n_features_after_selection": len(selected_features),
            "top_k_after_sparse": top_k_after_sparse,
        })

        fold_pred_df = pd.DataFrame({
            "Patient": X_va_model.index.astype(str),
            "outer_fold": outer_fold,
            "y_true": y_va.values,
            "y_pred_oof": pred_va,
            "clinical_mode": clinical_mode,
            "feature_selection_mode": feature_selection_mode,
            "tissue": tissue,
            "combo": combo,
            "tp": tp,
            "best_model_type": fit_result["best_model_type"],
            "top_k_after_sparse": top_k_after_sparse,
        })
        oof_rows.append(fold_pred_df)

        search_df = fit_result["search_df"].copy()
        search_df["outer_fold"] = outer_fold
        search_df["clinical_mode"] = clinical_mode
        search_df["feature_selection_mode_outer"] = feature_selection_mode
        search_df["tissue"] = tissue
        search_df["combo"] = combo
        search_df["tp"] = tp
        search_dfs.append(search_df)

        feature_info_df = extract_final_feature_info(best_pipe).copy()
        feature_info_df["outer_fold"] = outer_fold
        feature_info_df["clinical_mode"] = clinical_mode
        feature_info_df["feature_selection_mode"] = feature_selection_mode
        feature_info_df["tissue"] = tissue
        feature_info_df["combo"] = combo
        feature_info_df["tp"] = tp
        feature_dfs.append(feature_info_df)

        fw_df = fit_result.get("feature_weight_df", pd.DataFrame()).copy()
        if isinstance(fw_df, pd.DataFrame) and len(fw_df) > 0:
            fw_df["outer_fold"] = outer_fold
            fw_df["clinical_mode"] = clinical_mode
            fw_df["feature_selection_mode"] = feature_selection_mode
            fw_df["tissue"] = tissue
            fw_df["combo"] = combo
            fw_df["tp"] = tp
            feature_weight_dfs.append(fw_df)

        fs_df = fit_result.get("feature_selection_df", pd.DataFrame()).copy()
        if isinstance(fs_df, pd.DataFrame) and len(fs_df) > 0:
            fs_df["outer_fold"] = outer_fold
            fs_df["clinical_mode"] = clinical_mode
            fs_df["feature_selection_mode"] = feature_selection_mode
            fs_df["tissue"] = tissue
            fs_df["combo"] = combo
            fs_df["tp"] = tp
            feature_selection_dfs.append(fs_df)

    valid_mask = oof_pred.notna()
    if valid_mask.sum() >= 2:
        oof_r2 = float(r2_score(y_all_train.loc[valid_mask], oof_pred.loc[valid_mask]))
        oof_mae = float(mean_absolute_error(y_all_train.loc[valid_mask], oof_pred.loc[valid_mask]))
    else:
        oof_r2 = np.nan
        oof_mae = np.nan

    fold_metrics_df = pd.DataFrame(fold_metric_rows)
    oof_pred_df = pd.concat(oof_rows, ignore_index=True) if len(oof_rows) > 0 else pd.DataFrame()
    search_all_df = pd.concat(search_dfs, ignore_index=True) if len(search_dfs) > 0 else pd.DataFrame()
    feature_all_df = pd.concat(feature_dfs, ignore_index=True) if len(feature_dfs) > 0 else pd.DataFrame()
    feature_weight_all_df = pd.concat(feature_weight_dfs, ignore_index=True) if len(feature_weight_dfs) > 0 else pd.DataFrame()
    feature_selection_all_df = pd.concat(feature_selection_dfs, ignore_index=True) if len(feature_selection_dfs) > 0 else pd.DataFrame()

    mean_valid_r2 = float(np.nanmean(fold_metrics_df["valid_r2"])) if len(fold_metrics_df) else np.nan
    std_valid_r2 = float(np.nanstd(fold_metrics_df["valid_r2"])) if len(fold_metrics_df) else np.nan
    mean_train_r2 = float(np.nanmean(fold_metrics_df["train_r2"])) if len(fold_metrics_df) else np.nan
    mean_valid_mae = float(np.nanmean(fold_metrics_df["valid_mae"])) if len(fold_metrics_df) else np.nan
    mean_train_mae = float(np.nanmean(fold_metrics_df["train_mae"])) if len(fold_metrics_df) else np.nan

    # Fit one final model on the full TRAIN_IDS set for tracing/deployment.
    # This model is not used to compute OOF R².
    X_full_f, _ = filter_sparse_features_train_test(
        X_train=X_all,
        X_valid=X_all,
        min_obs_frac=min_obs_frac,
        protected_keep_cols=protected_base_cols,
    )

    if clinical_mode == "light":
        X_full_f, _, encoded_clin_cols_full = encode_clinical_train_valid(
            X_train=X_full_f,
            X_valid=X_full_f,
        )
        protected_keep_cols_full = encoded_clin_cols_full
    else:
        protected_keep_cols_full = []

    final_fit_result = fit_best_model_only(
        X_train=X_full_f,
        y_train=y_all_train,
        groups_train=X_full_f.index.to_numpy(),
        candidate_models=candidate_models,
        n_splits_inner=N_SPLITS_INNER,
        protected_keep_cols=protected_keep_cols_full,
        feature_selection_mode=feature_selection_mode,
        top_k_after_sparse=top_k_after_sparse,
    )

    final_selected = final_fit_result["selected_features"]
    X_full_model = X_full_f.loc[:, final_selected].copy()
    final_pipe = final_fit_result["fitted_pipeline"]
    final_train_pred = predict_1d(final_pipe, X_full_model)

    final_train_r2 = float(r2_score(y_all_train, final_train_pred)) if len(y_all_train) >= 2 else np.nan
    final_train_mae = float(mean_absolute_error(y_all_train, final_train_pred))

    final_train_pred_df = pd.DataFrame({
        "Patient": X_full_model.index.astype(str),
        "y_true": y_all_train.values,
        "y_pred_final_train_model": final_train_pred,
        "clinical_mode": clinical_mode,
        "feature_selection_mode": feature_selection_mode,
        "tissue": tissue,
        "combo": combo,
        "tp": tp,
        "best_model_type": final_fit_result["best_model_type"],
        "top_k_after_sparse": top_k_after_sparse,
    })

    final_feature_df = extract_final_feature_info(final_pipe).copy()

    metrics = {
        "selection_basis": "outer_cv_oof_r2_train_only_fold_safe_feature_selection",
        "outer_cv_type": outer_cv_type,
        "n_outer_folds": len(outer_splits),
        "n_train": len(X_all),
        "n_features_input": X.shape[1],
        "n_delta_features_input": len(delta_cols),
        "n_clinical_input": len(clin_cols_present),
        "oof_r2": oof_r2,
        "oof_mae": oof_mae,
        "mean_valid_r2": mean_valid_r2,
        "std_valid_r2": std_valid_r2,
        "mean_train_r2": mean_train_r2,
        "mean_valid_mae": mean_valid_mae,
        "mean_train_mae": mean_train_mae,
        "generalization_gap_mean_train_minus_valid": mean_train_r2 - mean_valid_r2,
        "final_full_train_best_model_type": final_fit_result["best_model_type"],
        "final_full_train_best_score_inner_oof_r2": final_fit_result["best_score"],
        "final_full_train_r2": final_train_r2,
        "final_full_train_mae": final_train_mae,
        "n_features_final_full_train": len(final_selected),
        "top_k_after_sparse": top_k_after_sparse,
    }

    return {
        "metrics": metrics,
        "fold_metrics_df": fold_metrics_df,
        "oof_pred_df": oof_pred_df,
        "search_df": search_all_df,
        "feature_df": feature_all_df,
        "feature_weight_df": feature_weight_all_df,
        "feature_selection_df": feature_selection_all_df,
        "final_train_pred_df": final_train_pred_df,
        "final_feature_df": final_feature_df,
    }


In [22]:
def run_one_single_omics_train_experiment(
    tissue: str,
    combo: str,
    tp: int,
    y_all: pd.Series,
    feature_mode: str,
    candidate_models: list[str],
    clinical_mode: str = "light",
    feature_selection_mode: str = "f_regression_topk",
    top_k_after_sparse_override: int | None = 100,
):
    feature_path = build_feature_path(
        feature_mode=feature_mode,
        tissue=tissue,
        combo=combo,
        tp=tp,
    )

    if not feature_path.exists():
        print("missing:", feature_path)
        return None

    print(
        f"\n[RUN-TRAIN-OOF] feature_mode={feature_mode}, "
        f"clinical_mode={clinical_mode}, selection={feature_selection_mode}, "
        f"tissue={tissue}, combo={combo}, tp={tp}"
    )
    print("feature_path:", feature_path)

    X = load_feature_csv(feature_path)
    raw_feature_signature = dataframe_content_signature(X)
    raw_feature_columns_hash = hashlib.md5("|".join(map(str, sorted(X.columns))).encode("utf-8")).hexdigest()
    X, y = align_xy(X, y_all)

    print("\n[EXPERIMENT START]")
    print("feature_mode:", feature_mode)
    print("clinical_mode:", clinical_mode)
    print("feature_selection_mode:", feature_selection_mode)
    print("tissue:", tissue, "| combo:", combo, "| tp:", tp)
    print("X shape before clinical:", X.shape)
    print("y shape:", y.shape)

    delta_cols = [c for c in X.columns if "_d" in str(c)]
    print("n_delta_features:", len(delta_cols))
    print("delta examples:", delta_cols[:10])

    if clinical_mode == "light":
        X = merge_with_light_clinical(
            X_feat=X,
            tissue=tissue,
            combo=combo,
            tp=tp,
        )
        assert_no_forbidden_proxy_columns(X, use_light_clinical=True)
    elif clinical_mode == "omics_only":
        assert_no_forbidden_proxy_columns(X, use_light_clinical=False)
    else:
        raise ValueError(f"Unknown clinical_mode: {clinical_mode}")

    delta_cols = [c for c in X.columns if "_d" in str(c)]
    clinical_cols_present = [c for c in get_clinical_columns() if c in X.columns]
    train_ids_in = [pid for pid in TRAIN_IDS if pid in X.index]

    base_skip = {
        "status": "skipped",
        "experiment_tag": EXPERIMENT_TAG,
        "use_feature_weighting": USE_FEATURE_WEIGHTING,
        "feature_weight_score_mode": FEATURE_WEIGHT_SCORE_MODE if USE_FEATURE_WEIGHTING else "none",
        "feature_weight_min": FEATURE_WEIGHT_MIN if USE_FEATURE_WEIGHTING else np.nan,
        "feature_weight_max": FEATURE_WEIGHT_MAX if USE_FEATURE_WEIGHTING else np.nan,
        "feature_mode": feature_mode,
        "clinical_mode": clinical_mode,
        "feature_selection_mode": feature_selection_mode,
        "tissue": tissue,
        "combo": combo,
        "tp": tp,
        "reason": None,
        "n_train_available": len(train_ids_in),
        "oof_r2": np.nan,
        "oof_mae": np.nan,
        "mean_valid_r2": np.nan,
        "std_valid_r2": np.nan,
        "mean_train_r2": np.nan,
        "mean_valid_mae": np.nan,
        "mean_train_mae": np.nan,
        "generalization_gap_mean_train_minus_valid": np.nan,
        "final_full_train_best_model_type": None,
        "final_full_train_best_score_inner_oof_r2": np.nan,
        "final_full_train_r2": np.nan,
        "final_full_train_mae": np.nan,
        "n_features_input": X.shape[1],
        "raw_feature_signature": raw_feature_signature,
        "raw_feature_columns_hash": raw_feature_columns_hash,
        "feature_path": str(feature_path),
        "n_delta_features_input": len(delta_cols),
        "n_clinical_input": len(clinical_cols_present),
        "n_features_final_full_train": np.nan,
        "top_k_after_sparse": top_k_after_sparse_override,
        "selection_basis": "not_run",
    }

    if len(train_ids_in) < N_SPLITS_OUTER:
        print("skip: not enough training patients after TRAIN_IDS filter")
        out = base_skip.copy()
        out["reason"] = "not enough training patients after TRAIN_IDS filter"
        return out

    top_k_after_sparse = top_k_after_sparse_override if top_k_after_sparse_override is not None else 100

    result = run_single_omics_train_oof_cv(
        X=X,
        y=y,
        tissue=tissue,
        combo=combo,
        tp=tp,
        train_ids=TRAIN_IDS,
        candidate_models=candidate_models,
        feature_mode=feature_mode,
        clinical_mode=clinical_mode,
        feature_selection_mode=feature_selection_mode,
        min_obs_frac=MIN_OBS_FRAC,
        top_k_after_sparse=top_k_after_sparse,
    )

    prefix = f"{feature_mode}_{clinical_mode}_{feature_selection_mode}_{tissue}_{combo}_tp{tp}_topk{top_k_after_sparse}"

    metrics = result["metrics"].copy()
    metrics.update({
        "status": "ok",
        "experiment_tag": EXPERIMENT_TAG,
        "use_feature_weighting": USE_FEATURE_WEIGHTING,
        "feature_weight_score_mode": FEATURE_WEIGHT_SCORE_MODE if USE_FEATURE_WEIGHTING else "none",
        "feature_weight_min": FEATURE_WEIGHT_MIN if USE_FEATURE_WEIGHTING else np.nan,
        "feature_weight_max": FEATURE_WEIGHT_MAX if USE_FEATURE_WEIGHTING else np.nan,
        "feature_mode": feature_mode,
        "clinical_mode": clinical_mode,
        "feature_selection_mode": feature_selection_mode,
        "tissue": tissue,
        "combo": combo,
        "tp": tp,
        "reason": None,
        "raw_feature_signature": raw_feature_signature,
        "raw_feature_columns_hash": raw_feature_columns_hash,
        "feature_path": str(feature_path),
    })

    pd.DataFrame([metrics]).to_csv(
        DETAIL_DIR / f"{prefix}_train_summary.csv",
        index=False
    )
    result["fold_metrics_df"].to_csv(
        DETAIL_DIR / f"{prefix}_train_fold_metrics.csv",
        index=False
    )
    result["oof_pred_df"].to_csv(
        DETAIL_DIR / f"{prefix}_train_oof_predictions.csv",
        index=False
    )
    result["search_df"].to_csv(
        DETAIL_DIR / f"{prefix}_train_search_results.csv",
        index=False
    )
    result["feature_df"].to_csv(
        DETAIL_DIR / f"{prefix}_train_outer_fold_features.csv",
        index=False
    )
    result["final_train_pred_df"].to_csv(
        DETAIL_DIR / f"{prefix}_final_full_train_predictions.csv",
        index=False
    )
    result["final_feature_df"].to_csv(
        DETAIL_DIR / f"{prefix}_final_full_train_features.csv",
        index=False
    )

    if isinstance(result.get("feature_weight_df", None), pd.DataFrame) and len(result["feature_weight_df"]) > 0:
        result["feature_weight_df"].to_csv(
            DETAIL_DIR / f"{prefix}_train_feature_weights.csv",
            index=False
        )

    if isinstance(result.get("feature_selection_df", None), pd.DataFrame) and len(result["feature_selection_df"]) > 0:
        result["feature_selection_df"].to_csv(
            DETAIL_DIR / f"{prefix}_train_feature_selection_scores.csv",
            index=False
        )

    return metrics


In [23]:
# ==========================================
# Sanity check before running Experiment B
# ==========================================
y_all = load_target(TARGET_PATH)

for tissue, combo, tp in product(TISSUES, COMBOS, TIMEPOINTS):
    feature_path = build_feature_path(feature_mode="omics", tissue=tissue, combo=combo, tp=tp)
    if not feature_path.exists():
        print("missing:", feature_path.name)
        continue

    X = load_feature_csv(feature_path)
    X, y = align_xy(X, y_all)
    
    

    delta_cols = [c for c in X.columns if "_d" in str(c)]
    print(
        f"{feature_path.name}: shape={X.shape}, delta_cols={len(delta_cols)}"
    )

    if len(delta_cols) > 0:
        print("  delta examples:", delta_cols[:10])

x_omics_csf_A_24.csv: shape=(103, 438), delta_cols=0
x_omics_csf_A_48.csv: shape=(103, 438), delta_cols=0
x_omics_csf_A_72.csv: shape=(103, 438), delta_cols=0
x_omics_csf_A_96.csv: shape=(103, 438), delta_cols=0
x_omics_csf_A_120.csv: shape=(103, 438), delta_cols=0
x_omics_csf_B_24.csv: shape=(83, 749), delta_cols=0
x_omics_csf_B_48.csv: shape=(83, 749), delta_cols=0
x_omics_csf_B_72.csv: shape=(83, 749), delta_cols=0
x_omics_csf_B_96.csv: shape=(83, 749), delta_cols=0
x_omics_csf_B_120.csv: shape=(83, 749), delta_cols=0
x_omics_csf_C_24.csv: shape=(39, 312), delta_cols=0
x_omics_csf_C_48.csv: shape=(39, 312), delta_cols=0
x_omics_csf_C_72.csv: shape=(39, 312), delta_cols=0
x_omics_csf_C_96.csv: shape=(39, 312), delta_cols=0
x_omics_csf_C_120.csv: shape=(39, 312), delta_cols=0
x_omics_ser_A_24.csv: shape=(108, 438), delta_cols=0
x_omics_ser_A_48.csv: shape=(108, 438), delta_cols=0
x_omics_ser_A_72.csv: shape=(108, 438), delta_cols=0
x_omics_ser_A_96.csv: shape=(108, 438), delta_cols=0


In [24]:
y_all = load_target(TARGET_PATH)

# Paper-style single-omics TRAIN-ONLY run.
# This produces outer-CV OOF results on TRAIN_IDS only.
# v11 rescue-expansion route table:
# - Keep all A/B/C combos and all 24/48/72/96/120h timepoints.
# - Restore v9-compatible candidate-model pools for critical routes.
# - Expand only high-potential route neighborhoods from v9/v10 OOF results.
# - The main comparable metric to the paper is OOF R², not train_r2/final_full_train_r2.
FEATURE_MODE_TO_RUN = "omics"


def _add_route(rows, *, tissue, combo, tp, top_k, feature_selection_mode, candidate_models, priority, reason):
    """Append one route while keeping candidate_models filtered to available model families."""
    candidate_models = [m for m in candidate_models if m in MODEL_TYPES]
    if len(candidate_models) == 0:
        return
    rows.append({
        "feature_mode": FEATURE_MODE_TO_RUN,
        "clinical_mode": "light",
        "feature_selection_mode": feature_selection_mode,
        "combo": combo,
        "top_k": int(top_k),
        "tissue": tissue,
        "tp": int(tp),
        "candidate_models": candidate_models,
        "priority": priority,
        "reason": reason,
    })


def _candidate_pool_v9_compatible(tissue: str, combo: str, tp: int) -> list[str]:
    """
    Important: this intentionally restores the v9-style candidate pools.
    The previous targeted version used many single-model routes, which made
    the critical v9 routes non-equivalent.
    """
    tissue = str(tissue).lower()
    combo = str(combo)
    tp = int(tp)

    if combo == "A":
        if tissue == "ser" and tp in [72, 96]:
            return ["svr_linear", "ridge", "elasticnet"]
        if tissue == "csf" and tp == 72:
            return ["svr_linear", "ridge", "gbr"]
        if tissue == "csf" and tp == 96:
            return ["svr_linear", "ridge", "elasticnet"]
        return ["ridge"]

    if combo == "B":
        if tissue == "ser" and tp in [96, 120]:
            return ["ridge", "elasticnet"]
        if tissue == "csf" and tp in [48, 72, 96]:
            return ["ridge", "elasticnet"]
        return ["ridge"]

    if combo == "C":
        if tissue == "ser" and tp == 24:
            return ["xgboost", "gbr", "ridge"]
        if tissue == "csf" and tp == 120:
            return ["ridge", "elasticnet", "pls"]
        return ["ridge"]

    return ["ridge"]


# Build explicit v11 rescue-expansion route table.
V11_ROUTE_ROWS = []

# -------------------------------
# A-family: strong/important single-omics signal.
# Start from v9 coverage and add targeted top-k expansion around strong regions.
# -------------------------------
A_BASE_TOPK = [40, 80, 120]
A_EXTRA_TOPK = {
    ("ser", 72): [100, 140],
    ("ser", 96): [100, 140, 160],
    ("csf", 72): [60, 100],
    ("csf", 96): [30, 60, 100],
}

for tissue in ["csf", "ser"]:
    for tp in [24, 48, 72, 96, 120]:
        pool = _candidate_pool_v9_compatible(tissue, "A", tp)
        topks = sorted(set(A_BASE_TOPK + A_EXTRA_TOPK.get((tissue, tp), [])))
        for top_k in topks:
            for fs in ["f_regression_topk", "corr_topk"]:
                is_critical = (tissue == "ser" and tp == 96 and top_k in [120, 140])
                is_high = (tissue == "ser" and tp in [72, 96]) or (tissue == "csf" and tp in [72, 96])
                _add_route(
                    V11_ROUTE_ROWS,
                    tissue=tissue,
                    combo="A",
                    tp=tp,
                    top_k=top_k,
                    feature_selection_mode=fs,
                    candidate_models=pool,
                    priority="critical" if is_critical else ("high" if is_high else "normal"),
                    reason="v11 A rescue-expansion: v9-compatible model pool + expanded top_k around strong A routes",
                )

# -------------------------------
# B-family: keep minimal coverage. B was weak, so do not spend the budget here.
# Use v9-compatible pools where B previously allowed elasticnet.
# -------------------------------
B_SELECTED = {
    ("csf", 24): ("f_regression_topk", 40),
    ("csf", 48): ("f_regression_topk", 40),
    ("csf", 72): ("f_regression_topk", 40),
    ("csf", 96): ("f_regression_topk", 40),
    ("csf", 120): ("f_regression_topk", 40),
    ("ser", 24): ("f_regression_topk", 40),
    ("ser", 48): ("f_regression_topk", 80),
    ("ser", 72): ("f_regression_topk", 40),
    ("ser", 96): ("f_regression_topk", 80),
    ("ser", 120): ("f_regression_topk", 40),
}
for (tissue, tp), (fs, top_k) in B_SELECTED.items():
    _add_route(
        V11_ROUTE_ROWS,
        tissue=tissue,
        combo="B",
        tp=tp,
        top_k=top_k,
        feature_selection_mode=fs,
        candidate_models=_candidate_pool_v9_compatible(tissue, "B", tp),
        priority="low" if (tissue, tp) in [("csf", 24), ("csf", 120), ("ser", 24), ("ser", 72)] else "normal",
        reason="v11 B minimal coverage: v9-compatible pool but no broad expansion",
    )

# -------------------------------
# C-family: rescue ser C 24h and expand csf C around new v10 targeted signal.
# -------------------------------
# ser C 24h: critical. Restore v9 candidate pool xgboost+gbr+ridge and expand top_k.
for top_k in [10, 20, 30, 40, 50, 60]:
    for fs in ["f_regression_topk", "corr_topk"]:
        _add_route(
            V11_ROUTE_ROWS,
            tissue="ser",
            combo="C",
            tp=24,
            top_k=top_k,
            feature_selection_mode=fs,
            candidate_models=_candidate_pool_v9_compatible("ser", "C", 24),
            priority="critical" if (fs == "f_regression_topk" and top_k in [30, 40, 50]) else "high",
            reason="v11 ser C 24h rescue: restore v9 candidate pool xgboost+gbr+ridge and expand top_k",
        )

# ser C non-24h: stable ridge route with small top_k neighborhood.
for tp in [48, 72, 96, 120]:
    for top_k in [5, 10, 15, 20]:
        _add_route(
            V11_ROUTE_ROWS,
            tissue="ser",
            combo="C",
            tp=tp,
            top_k=top_k,
            feature_selection_mode="f_regression_topk",
            candidate_models=_candidate_pool_v9_compatible("ser", "C", tp),
            priority="normal" if top_k == 10 else "low",
            reason="v11 ser C non-24h: ridge-focused top_k neighborhood around v9 stable top10 route",
        )

# csf C: include corr_topk because v10 targeted found csf C 24h corr_topk top20 as a strong route.
for tp in [24, 48, 72, 96, 120]:
    for top_k in [15, 20, 30]:
        for fs in ["f_regression_topk", "corr_topk"]:
            _add_route(
                V11_ROUTE_ROWS,
                tissue="csf",
                combo="C",
                tp=tp,
                top_k=top_k,
                feature_selection_mode=fs,
                candidate_models=_candidate_pool_v9_compatible("csf", "C", tp),
                priority="high" if (tp in [24, 120] and top_k == 20) else "normal",
                reason="v11 csf C expansion: include corr_topk and top_k neighborhood around 20",
            )

# Deduplicate defensively.
_seen = set()
_deduped = []
for row in V11_ROUTE_ROWS:
    key = (
        row["feature_mode"], row["clinical_mode"], row["feature_selection_mode"],
        row["combo"], row["top_k"], row["tissue"], row["tp"], tuple(row["candidate_models"]),
    )
    if key not in _seen:
        _seen.add(key)
        _deduped.append(row)
V11_ROUTE_ROWS = _deduped

jobs = V11_ROUTE_ROWS
print("n_jobs_to_run:", len(jobs))
job_manifest_df = pd.DataFrame(jobs)

# Coverage check: should be 30 = 2 tissues * 3 combos * 5 TPs.
coverage_df = (
    job_manifest_df[["tissue", "combo", "tp"]]
    .drop_duplicates()
    .sort_values(["tissue", "combo", "tp"])
    .reset_index(drop=True)
)
print("coverage rows:", len(coverage_df), "| expected:", len(TISSUES) * len(COMBOS) * len(TIMEPOINTS))
missing_coverage = []
for tissue in TISSUES:
    for combo in COMBOS:
        for tp in TIMEPOINTS:
            mask = (
                (coverage_df["tissue"] == tissue)
                & (coverage_df["combo"] == combo)
                & (coverage_df["tp"] == tp)
            )
            if not mask.any():
                missing_coverage.append((tissue, combo, tp))
print("missing coverage:", missing_coverage)
if len(missing_coverage) > 0:
    raise ValueError(f"Missing required tissue/combo/TP coverage: {missing_coverage}")

display(job_manifest_df.head(20))
print("candidate model counts:")
display(job_manifest_df["candidate_models"].astype(str).value_counts().reset_index(name="n_jobs"))
print("route priority counts:")
display(job_manifest_df["priority"].value_counts().reset_index(name="n_jobs"))
print("route counts by tissue/combo/priority:")
display(job_manifest_df.groupby(["tissue", "combo", "priority"]).size().reset_index(name="n_routes"))

job_manifest_path = SAVE_DIR / f"job_manifest_{EXPERIMENT_TAG}.csv"
coverage_check_path = SAVE_DIR / f"coverage_check_{EXPERIMENT_TAG}.csv"
route_summary_path = SAVE_DIR / f"route_summary_{EXPERIMENT_TAG}.csv"

job_manifest_df.to_csv(job_manifest_path, index=False)
coverage_df.to_csv(coverage_check_path, index=False)
job_manifest_df.groupby(["tissue", "combo", "priority"]).size().reset_index(name="n_routes").to_csv(route_summary_path, index=False)

print("saved job manifest:", job_manifest_path)
print("saved coverage check:", coverage_check_path)
print("saved route summary:", route_summary_path)

print("feature-selection routing counts:")
display(job_manifest_df["feature_selection_mode"].value_counts().reset_index(name="n_jobs"))

def run_one_single_omics_train_experiment_safe(job: dict, y_all: pd.Series):
    feature_mode = job["feature_mode"]
    clinical_mode = job["clinical_mode"]
    feature_selection_mode = job["feature_selection_mode"]
    tissue = job["tissue"]
    combo = job["combo"]
    tp = job["tp"]
    top_k = job["top_k"]
    candidate_models = job.get("candidate_models", MODEL_TYPES)

    try:
        result = run_one_single_omics_train_experiment(
            tissue=tissue,
            combo=combo,
            tp=tp,
            y_all=y_all,
            feature_mode=feature_mode,
            candidate_models=candidate_models,
            clinical_mode=clinical_mode,
            feature_selection_mode=feature_selection_mode,
            top_k_after_sparse_override=top_k,
        )
        if isinstance(result, dict):
            result["route_priority"] = job.get("priority")
            result["route_reason"] = job.get("reason")
        return result

    except Exception as e:
        print(
            f"[ERROR] feature_mode={feature_mode}, clinical={clinical_mode}, "
            f"selection={feature_selection_mode}, tissue={tissue}, "
            f"combo={combo}, tp={tp}, top_k={top_k} :: {e}"
        )
        return {
            "status": "failed",
            "experiment_tag": EXPERIMENT_TAG,
            "use_feature_weighting": USE_FEATURE_WEIGHTING,
            "feature_weight_score_mode": FEATURE_WEIGHT_SCORE_MODE if USE_FEATURE_WEIGHTING else "none",
            "feature_weight_min": FEATURE_WEIGHT_MIN if USE_FEATURE_WEIGHTING else np.nan,
            "feature_weight_max": FEATURE_WEIGHT_MAX if USE_FEATURE_WEIGHTING else np.nan,
            "feature_mode": feature_mode,
            "clinical_mode": clinical_mode,
            "feature_selection_mode": feature_selection_mode,
            "tissue": tissue,
            "combo": combo,
            "tp": tp,
            "reason": str(e),
            "route_priority": job.get("priority"),
            "route_reason": job.get("reason"),
            "n_train": np.nan,
            "oof_r2": np.nan,
            "oof_mae": np.nan,
            "mean_valid_r2": np.nan,
            "std_valid_r2": np.nan,
            "mean_train_r2": np.nan,
            "mean_valid_mae": np.nan,
            "mean_train_mae": np.nan,
            "generalization_gap_mean_train_minus_valid": np.nan,
            "final_full_train_best_model_type": None,
            "final_full_train_best_score_inner_oof_r2": np.nan,
            "final_full_train_r2": np.nan,
            "final_full_train_mae": np.nan,
            "n_features_input": np.nan,
            "raw_feature_signature": None,
            "raw_feature_columns_hash": None,
            "feature_path": str(build_feature_path(feature_mode, tissue, combo, tp)),
            "n_delta_features_input": np.nan,
            "n_clinical_input": np.nan,
            "n_features_final_full_train": np.nan,
            "top_k_after_sparse": top_k,
            "candidate_models": ",".join(candidate_models),
            "selection_basis": "outer_cv_oof_r2_train_only_fold_safe_feature_selection",
        }


rows = Parallel(
    n_jobs=N_JOBS_EXPERIMENT,
    backend="loky",
    verbose=10,
)(
    delayed(run_one_single_omics_train_experiment_safe)(job, y_all)
    for job in jobs
)

summary_df = pd.DataFrame(rows)

sort_cols = [
    "oof_r2", "mean_valid_r2", "mean_train_r2", "oof_mae",
    "final_full_train_best_score_inner_oof_r2",
]
for c in sort_cols:
    if c not in summary_df.columns:
        summary_df[c] = np.nan

if len(summary_df) > 0:
    summary_df = summary_df.sort_values(
        by=["status", "oof_r2", "mean_valid_r2"],
        ascending=[True, False, False],
        na_position="last",
    ).reset_index(drop=True)

out_path = SAVE_DIR / f"{FEATURE_MODE_TO_RUN}_paper_style_train_summary_{EXPERIMENT_TAG}.csv"
summary_df.to_csv(out_path, index=False)

print("saved:", out_path)

# Grouped summaries for OOF-R² analysis.
ok_df = summary_df[summary_df["status"].eq("ok")].copy()


def save_grouped_summary(df: pd.DataFrame, group_cols: list[str], name: str):
    if len(df) == 0:
        return None
    grouped = (
        df.groupby(group_cols, dropna=False)
          .agg(
              n=("oof_r2", "size"),
              mean_oof_r2=("oof_r2", "mean"),
              median_oof_r2=("oof_r2", "median"),
              best_oof_r2=("oof_r2", "max"),
              mean_valid_r2=("mean_valid_r2", "mean"),
              mean_train_r2=("mean_train_r2", "mean"),
              mean_gap=("generalization_gap_mean_train_minus_valid", "mean"),
          )
          .reset_index()
          .sort_values(["mean_oof_r2", "best_oof_r2"], ascending=False)
    )
    path = SAVE_DIR / f"{name}_{EXPERIMENT_TAG}.csv"
    grouped.to_csv(path, index=False)
    print("saved grouped summary:", path)
    return grouped


grouped_by_tissue = save_grouped_summary(ok_df, ["tissue"], "grouped_by_tissue")
grouped_by_tissue_combo = save_grouped_summary(ok_df, ["tissue", "combo"], "grouped_by_tissue_combo")
grouped_by_tissue_combo_tp = save_grouped_summary(ok_df, ["tissue", "combo", "tp"], "grouped_by_tissue_combo_tp")
grouped_by_tissue_combo_model = save_grouped_summary(ok_df, ["tissue", "combo", "final_full_train_best_model_type"], "grouped_by_tissue_combo_model")
grouped_by_tissue_combo_tp_model = save_grouped_summary(ok_df, ["tissue", "combo", "tp", "final_full_train_best_model_type"], "grouped_by_tissue_combo_tp_model")
grouped_by_tissue_combo_tp_selection = save_grouped_summary(ok_df, ["tissue", "combo", "tp", "feature_selection_mode"], "grouped_by_tissue_combo_tp_selection")

# Paper-style best row per tissue + combo + TP.
# This is the main table for comparing single-omics results by TP rather than using one global mean.
if len(ok_df) > 0:
    best_by_tissue_combo_tp = (
        ok_df.sort_values(["tissue", "combo", "tp", "oof_r2", "mean_valid_r2"], ascending=[True, True, True, False, False])
             .groupby(["tissue", "combo", "tp"], as_index=False)
             .first()
             .sort_values(["tissue", "combo", "tp"])
    )
    best_path = SAVE_DIR / f"paper_style_best_by_tissue_combo_tp_{EXPERIMENT_TAG}.csv"
    best_by_tissue_combo_tp.to_csv(best_path, index=False)
    print("saved paper-style best by tissue+combo+tp:", best_path)

    best_by_tissue_tp = (
        ok_df.sort_values(["tissue", "tp", "oof_r2", "mean_valid_r2"], ascending=[True, True, False, False])
             .groupby(["tissue", "tp"], as_index=False)
             .first()
             .sort_values(["tissue", "tp"])
    )
    best_tp_path = SAVE_DIR / f"paper_style_best_by_tissue_tp_{EXPERIMENT_TAG}.csv"
    best_by_tissue_tp.to_csv(best_tp_path, index=False)
    print("saved paper-style best by tissue+tp:", best_tp_path)

# Diagnostic: identify feature matrices that are identical across different TPs.
if "raw_feature_signature" in ok_df.columns:
    duplicate_feature_matrix_check = (
        ok_df.groupby(["tissue", "combo", "raw_feature_signature"], dropna=False)
             .agg(
                 n_rows=("oof_r2", "size"),
                 timepoints=("tp", lambda s: ",".join(map(str, sorted(set(s))))),
                 n_unique_tp=("tp", lambda s: len(set(s))),
                 feature_path_examples=("feature_path", lambda s: " | ".join(map(str, list(s)[:3]))),
             )
             .reset_index()
             .query("n_unique_tp > 1")
             .sort_values(["tissue", "combo", "n_unique_tp"], ascending=[True, True, False])
    )
    dup_path = SAVE_DIR / f"duplicate_feature_matrix_check_{EXPERIMENT_TAG}.csv"
    duplicate_feature_matrix_check.to_csv(dup_path, index=False)
    print("saved duplicate feature-matrix diagnostic:", dup_path)
    display(duplicate_feature_matrix_check.head(20))

print("\nTop 30 by TRAIN OOF R²:")
display_cols = [
    "status", "clinical_mode", "feature_selection_mode", "tissue", "combo", "tp",
    "top_k_after_sparse", "final_full_train_best_model_type",
    "oof_r2", "oof_mae", "mean_valid_r2", "std_valid_r2",
    "mean_train_r2", "generalization_gap_mean_train_minus_valid",
    "final_full_train_best_score_inner_oof_r2",
    "n_features_final_full_train",
]
optional_cols = ["route_priority", "route_reason"]
display_cols_existing = [c for c in display_cols + optional_cols if c in summary_df.columns]
display(summary_df[display_cols_existing].head(30))

print("\nBest by clinical/selection/tissue/combo/tp:")
best_by_group = (
    summary_df[summary_df["status"].eq("ok")]
    .sort_values("oof_r2", ascending=False)
    .groupby(["clinical_mode", "feature_selection_mode", "tissue", "combo", "tp"], as_index=False)
    .first()
)
display(best_by_group[[c for c in display_cols_existing if c in best_by_group.columns]].head(50))




n_jobs_to_run: 148
coverage rows: 30 | expected: 30
missing coverage: []


,feature_mode,clinical_mode,feature_selection_mode,combo,top_k,tissue,tp,candidate_models,priority,reason
0,omics,light,f_regression_topk,A,40,csf,24,[ridge],normal,v11 A rescue-expansion: v9-compatible model po...
1,omics,light,corr_topk,A,40,csf,24,[ridge],normal,v11 A rescue-expansion: v9-compatible model po...
2,omics,light,f_regression_topk,A,80,csf,24,[ridge],normal,v11 A rescue-expansion: v9-compatible model po...
3,omics,light,corr_topk,A,80,csf,24,[ridge],normal,v11 A rescue-expansion: v9-compatible model po...
4,omics,light,f_regression_topk,A,120,csf,24,[ridge],normal,v11 A rescue-expansion: v9-compatible model po...
5,omics,light,corr_topk,A,120,csf,24,[ridge],normal,v11 A rescue-expansion: v9-compatible model po...
6,omics,light,f_regression_topk,A,40,csf,48,[ridge],normal,v11 A rescue-expansion: v9-compatible model po...
7,omics,light,corr_topk,A,40,csf,48,[ridge],normal,v11 A rescue-expansion: v9-compatible model po...
8,omics,light,f_regression_topk,A,80,csf,48,[ridge],normal,v11 A rescue-expansion: v9-compatible model po...
9,omics,light,corr_topk,A,80,csf,48,[ridge],normal,v11 A rescue-expansion: v9-compatible model po...


candidate model counts:


,candidate_models,n_jobs
0,['ridge'],81
1,"['svr_linear', 'ridge', 'elasticnet']",34
2,"['xgboost', 'gbr', 'ridge']",12
3,"['svr_linear', 'ridge', 'gbr']",10
4,"['ridge', 'elasticnet', 'pls']",6
5,"['ridge', 'elasticnet']",5


route priority counts:


...

,tissue,combo,raw_feature_signature,n_rows,timepoints,n_unique_tp,feature_path_examples



Top 30 by TRAIN OOF R²:


,status,clinical_mode,feature_selection_mode,tissue,combo,tp,top_k_after_sparse,final_full_train_best_model_type,oof_r2,oof_mae,mean_valid_r2,std_valid_r2,mean_train_r2,generalization_gap_mean_train_minus_valid,final_full_train_best_score_inner_oof_r2,n_features_final_full_train,route_priority,route_reason
0,ok,light,f_regression_topk,ser,C,24,40,xgboost,0.271758,15.176796,0.194197,0.176020,0.733303,0.539106,0.176393,40,critical,v11 ser C 24h rescue: restore v9 candidate poo...
1,ok,light,corr_topk,csf,C,24,30,ridge,0.261029,14.235372,0.240782,0.225431,0.797261,0.556479,0.159739,30,normal,v11 csf C expansion: include corr_topk and top...
2,ok,light,corr_topk,csf,C,24,20,ridge,0.253265,13.439060,0.316194,0.200290,0.674126,0.357932,0.276462,20,high,v11 csf C expansion: include corr_topk and top...
3,ok,light,corr_topk,ser,A,96,120,svr_linear,0.242782,13.388230,0.306499,0.302850,0.683036,0.376537,0.161328,120,critical,v11 A rescue-expansion: v9-compatible model po...
4,ok,light,f_regression_topk,csf,A,96,30,ridge,0.236085,13.732454,0.199354,0.296724,0.501134,0.301780,0.212344,30,high,v11 A rescue-expansion: v9-compatible model po...
5,ok,light,f_regression_topk,ser,A,96,100,svr_linear,0.234564,13.494215,0.288056,0.272773,0.634685,0.346629,0.176559,100,high,v11 A rescue-expansion: v9-compatible model po...
6,ok,light,corr_topk,ser,A,96,100,svr_linear,0.231528,13.546047,0.283952,0.271167,0.636724,0.352771,0.178925,100,high,v11 A rescue-expansion: v9-compatible model po...
7,ok,light,corr_topk,csf,A,96,30,ridge,0.226360,13.697032,0.188781,0.303123,0.517226,0.328445,0.203671,30,high,v11 A rescue-expansion: v9-compatible model po...
8,ok,light,f_regression_topk,ser,A,96,120,svr_linear,0.217384,13.474596,0.270970,0.290776,0.660641,0.389671,0.170774,120,critical,v11 A rescue-expansion: v9-compatible model po...
9,ok,light,corr_topk,csf,C,48,30,ridge,0.201590,14.601400,0.186037,0.241039,0.696185,0.510148,0.167297,30,normal,v11 csf C expansion: include corr_topk and top...



Best by clinical/selection/tissue/combo/tp:


,status,clinical_mode,feature_selection_mode,tissue,combo,tp,top_k_after_sparse,final_full_train_best_model_type,oof_r2,oof_mae,mean_valid_r2,std_valid_r2,mean_train_r2,generalization_gap_mean_train_minus_valid,final_full_train_best_score_inner_oof_r2,n_features_final_full_train,route_priority,route_reason
0,ok,light,corr_topk,csf,A,24,120,ridge,0.059446,15.369274,0.068281,0.253235,0.574360,0.506079,-0.007916,120,normal,v11 A rescue-expansion: v9-compatible model po...
1,ok,light,corr_topk,csf,A,48,40,ridge,0.016371,15.672716,-0.003948,0.418752,0.595960,0.599908,0.008625,40,normal,v11 A rescue-expansion: v9-compatible model po...
2,ok,light,corr_topk,csf,A,72,60,gbr,0.195500,14.211108,0.126244,0.287160,0.562045,0.435801,0.113992,60,high,v11 A rescue-expansion: v9-compatible model po...
3,ok,light,corr_topk,csf,A,96,30,ridge,0.226360,13.697032,0.188781,0.303123,0.517226,0.328445,0.203671,30,high,v11 A rescue-expansion: v9-compatible model po...
4,ok,light,corr_topk,csf,A,120,120,ridge,0.028866,15.770312,0.015004,0.116635,0.490056,0.475052,0.044072,120,normal,v11 A rescue-expansion: v9-compatible model po...
5,ok,light,corr_topk,csf,C,24,30,ridge,0.261029,14.235372,0.240782,0.225431,0.797261,0.556479,0.159739,30,normal,v11 csf C expansion: include corr_topk and top...
6,ok,light,corr_topk,csf,C,48,30,ridge,0.201590,14.601400,0.186037,0.241039,0.696185,0.510148,0.167297,30,normal,v11 csf C expansion: include corr_topk and top...
7,ok,light,corr_topk,csf,C,72,30,ridge,0.090519,15.485917,0.067422,0.145476,0.684612,0.617191,-0.066097,30,normal,v11 csf C expansion: include corr_topk and top...
8,ok,light,corr_topk,csf,C,96,30,ridge,0.014829,15.663919,-0.071074,0.364555,0.624650,0.695724,0.095775,30,normal,v11 csf C expansion: include corr_topk and top...
9,ok,light,corr_topk,csf,C,120,30,ridge,0.181183,14.333944,0.158403,0.162130,0.583065,0.424662,0.096037,30,normal,v11 csf C expansion: include corr_topk and top...


In [25]:
print("SAVE_DIR:", SAVE_DIR.resolve())
print("DETAIL_DIR:", DETAIL_DIR.resolve())
print("FEATURE_TRACE_DIR:", FEATURE_TRACE_DIR.resolve())


SAVE_DIR: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/results_single_omics_train_only_single_omics_train_paper_style_v11_rescue_expansion_v9compatible
DETAIL_DIR: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/results_single_omics_train_only_single_omics_train_paper_style_v11_rescue_expansion_v9compatible/single_detail_outputs
FEATURE_TRACE_DIR: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/results_single_omics_train_only_single_omics_train_paper_style_v11_rescue_expansion_v9compatible/feature_traces
